In [ ]:
import os

Key here

In [ ]:
!pip install -U transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.6 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [ ]:
!pip install -U "bitsandbytes>=0.46.1" transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [ ]:
import torch

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

CUDA available: True
GPU: Tesla T4


In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


def main():

    # ---------------------------------------------------------
    # 1. Check GPU
    # ---------------------------------------------------------

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is required for this QLoRA exercise."
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    # ---------------------------------------------------------
    # 2. Model
    # ---------------------------------------------------------

    model_name = (
        "Qwen/Qwen2.5-1.5B-Instruct"
    )

    # ---------------------------------------------------------
    # 3. Decide computation precision
    # ---------------------------------------------------------

    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    print(
        "Compute dtype:",
        compute_dtype
    )

    # ---------------------------------------------------------
    # 4. Configure 4-bit quantization
    # ---------------------------------------------------------

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    # ---------------------------------------------------------
    # 5. Load tokenizer
    # ---------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    # ---------------------------------------------------------
    # 6. Load Qwen in 4-bit
    # ---------------------------------------------------------

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
    )

    print(
        "\nBase model loaded."
    )

    print(
        "Memory footprint:",
        round(
            model.get_memory_footprint()
            / 1024**3,
            2,
        ),
        "GB",
    )

    # ---------------------------------------------------------
    # 7. Prepare quantized model for training
    # ---------------------------------------------------------

    model = prepare_model_for_kbit_training(
        model
    )

    # ---------------------------------------------------------
    # 8. Configure LoRA
    # ---------------------------------------------------------

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )

    # ---------------------------------------------------------
    # 9. Attach LoRA adapters
    # ---------------------------------------------------------

    model = get_peft_model(
        model,
        lora_config,
    )

    # ---------------------------------------------------------
    # 10. Show trainable parameters
    # ---------------------------------------------------------

    print(
        "\nTRAINABLE PARAMETERS"
    )

    print(
        "=" * 70
    )

    model.print_trainable_parameters()

    # ---------------------------------------------------------
    # 11. Show a few trainable parameter names
    # ---------------------------------------------------------

    print(
        "\nSAMPLE TRAINABLE PARAMETERS"
    )

    print(
        "=" * 70
    )

    count = 0

    for name, parameter in model.named_parameters():

        if parameter.requires_grad:

            print(
                name,
                parameter.shape,
            )

            count += 1

            if count == 5:
                break


if __name__ == "__main__":
    main()

GPU: Tesla T4
Compute dtype: torch.bfloat16


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Base model loaded.
Memory footprint: 1.05 GB

TRAINABLE PARAMETERS
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

SAMPLE TRAINABLE PARAMETERS
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([16, 1536])
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([1536, 16])
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([16, 1536])
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([256, 16])
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([16, 1536])


In [ ]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer


def main():

    model_name = "Qwen/Qwen2.5-1.5B-Instruct"

    # ---------------------------------------------------------
    # 1. Raw curated products
    # ---------------------------------------------------------

    products = [
        {
            "description": "USB-C fast charging cable",
            "price": 15,
        },
        {
            "description": "Portable desk fan",
            "price": 25,
        },
        {
            "description": "Stainless steel coffee maker",
            "price": 130,
        },
        {
            "description":
                "Premium wireless noise cancelling headphones",
            "price": 275,
        },
        {
            "description": "27 inch 4K gaming monitor",
            "price": 450,
        },
        {
            "description":
                "65 inch OLED 4K smart television",
            "price": 850,
        },
    ]

    # ---------------------------------------------------------
    # 2. Convert products into chat training examples
    # ---------------------------------------------------------

    training_examples = []

    for product in products:

        example = {
            "messages": [
                {
                    "role": "system",
                    "content":
                        "You estimate product prices. "
                        "Reply only with the price "
                        "in US dollars.",
                },
                {
                    "role": "user",
                    "content":
                        product["description"],
                },
                {
                    "role": "assistant",
                    "content":
                        f"${product['price']}",
                },
            ]
        }

        training_examples.append(
            example
        )

    # ---------------------------------------------------------
    # 3. Convert Python list → Hugging Face Dataset
    # ---------------------------------------------------------

    dataset = Dataset.from_list(
        training_examples
    )

    print("\nDATASET")
    print("=" * 70)

    print(dataset)

    print(
        "\nNumber of examples:",
        len(dataset)
    )

    # ---------------------------------------------------------
    # 4. Inspect first structured example
    # ---------------------------------------------------------

    first_example = dataset[0]

    print("\nFIRST RAW EXAMPLE")
    print("=" * 70)

    print(
        first_example
    )

    # ---------------------------------------------------------
    # 5. Load Qwen tokenizer
    # ---------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    # ---------------------------------------------------------
    # 6. Apply Qwen's chat template
    # ---------------------------------------------------------

    formatted_text = tokenizer.apply_chat_template(
        first_example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    print("\nFORMATTED CHAT")
    print("=" * 70)

    print(
        formatted_text
    )

    # ---------------------------------------------------------
    # 7. Tokenize using the chat template directly
    # ---------------------------------------------------------

    token_ids = tokenizer.apply_chat_template(
        first_example["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )

    print("\nTOKEN IDS")
    print("=" * 70)

    print(
        token_ids
    )

    print(
        "\nNumber of tokens:",
        len(token_ids)
    )

    # ---------------------------------------------------------
    # 8. Decode tokens back into text
    # ---------------------------------------------------------

    decoded_text = tokenizer.decode(
        token_ids
    )

    print("\nDECODED TOKENS")
    print("=" * 70)

    print(
        decoded_text
    )


if __name__ == "__main__":
    main()


DATASET
Dataset({
    features: ['messages'],
    num_rows: 6
})

Number of examples: 6

FIRST RAW EXAMPLE
{'messages': [{'content': 'You estimate product prices. Reply only with the price in US dollars.', 'role': 'system'}, {'content': 'USB-C fast charging cable', 'role': 'user'}, {'content': '$15', 'role': 'assistant'}]}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


FORMATTED CHAT
<|im_start|>system
You estimate product prices. Reply only with the price in US dollars.<|im_end|>
<|im_start|>user
USB-C fast charging cable<|im_end|>
<|im_start|>assistant
$15<|im_end|>


TOKEN IDS
{'input_ids': [151644, 8948, 198, 2610, 16045, 1985, 7576, 13, 17841, 1172, 448, 279, 3349, 304, 2274, 11192, 13, 151645, 198, 151644, 872, 198, 26032, 7658, 4937, 22641, 14652, 151645, 198, 151644, 77091, 198, 3, 16, 20, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Number of tokens: 2

DECODED TOKENS
<|im_start|>system
You estimate product prices. Reply only with the price in US dollars.<|im_end|>
<|im_start|>user
USB-C fast charging cable<|im_end|>
<|im_start|>assistant
$15<|im_end|>



In [3]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 24.5 MB/s eta 0:00:00


In [4]:
!pip install -U transformers datasets peft trl accelerate "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalli

In [ ]:
import json
import os
import random
import re

import requests


# -------------------------------------------------------------
# Configuration
# -------------------------------------------------------------

DATA_URL = (
    "https://huggingface.co/datasets/"
    "McAuley-Lab/Amazon-Reviews-2023/"
    "resolve/main/raw/meta_categories/"
    "meta_Appliances.jsonl"
)


TRAIN_SIZE = 500
VALIDATION_SIZE = 50
TEST_SIZE = 250


TOTAL_REQUIRED = (
    TRAIN_SIZE
    + VALIDATION_SIZE
    + TEST_SIZE
)


MIN_PRICE = 0.50
MAX_PRICE = 999.49


MAX_DESCRIPTION_CHARS = 2500


RANDOM_SEED = 42


TRAIN_FILE = "amazon_price_train.jsonl"

VALIDATION_FILE = (
    "amazon_price_validation.jsonl"
)

TEST_FILE = "amazon_price_test.jsonl"


# -------------------------------------------------------------
# Parse price
# -------------------------------------------------------------

def parse_price(value):

    if value is None:
        return None

    text = str(value).strip()

    if not text:
        return None

    text = text.replace(
        "$",
        "",
    )

    text = text.replace(
        ",",
        "",
    )

    match = re.search(
        r"\d+(?:\.\d+)?",
        text,
    )

    if not match:
        return None

    try:

        return float(
            match.group()
        )

    except ValueError:

        return None


# -------------------------------------------------------------
# Clean text
# -------------------------------------------------------------

def clean_text(value):

    if value is None:
        return ""

    text = str(value)

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


# -------------------------------------------------------------
# Convert list/string into normal text
# -------------------------------------------------------------

def join_text(value):

    if value is None:
        return ""

    if isinstance(
        value,
        list,
    ):

        cleaned_items = []

        for item in value:

            cleaned = clean_text(
                item
            )

            if cleaned:

                cleaned_items.append(
                    cleaned
                )

        return " ".join(
            cleaned_items
        )

    return clean_text(
        value
    )


# -------------------------------------------------------------
# Build product text
#
# We use:
#
# title
# +
# features
# +
# description
# -------------------------------------------------------------

def build_product_text(product):

    title = clean_text(
        product.get(
            "title"
        )
    )

    features = join_text(
        product.get(
            "features"
        )
    )

    description = join_text(
        product.get(
            "description"
        )
    )

    parts = []

    if title:

        parts.append(
            title
        )

    if features:

        parts.append(
            features
        )

    if description:

        parts.append(
            description
        )

    combined = " ".join(
        parts
    )

    combined = clean_text(
        combined
    )

    # ---------------------------------------------------------
    # Character-level truncation.
    #
    # Token-level truncation happens later during SFT.
    # ---------------------------------------------------------

    return combined[
        :MAX_DESCRIPTION_CHARS
    ]


# -------------------------------------------------------------
# Curate one Amazon product
# -------------------------------------------------------------

def curate_product(product):

    price = parse_price(
        product.get(
            "price"
        )
    )

    if price is None:

        return None

    if not (
        MIN_PRICE
        <= price
        <= MAX_PRICE
    ):

        return None

    description = build_product_text(
        product
    )

    # Very short descriptions contain little useful signal.
    if len(description) < 40:

        return None

    product_id = product.get(
        "parent_asin"
    )

    if product_id is None:

        product_id = ""

    return {
        "description": description,

        "price": round(
            price,
            2,
        ),

        "product_id": str(
            product_id
        ),
    }


# -------------------------------------------------------------
# Write JSONL
# -------------------------------------------------------------

def write_jsonl(
    filename,
    products,
):

    with open(
        filename,
        "w",
        encoding="utf-8",
    ) as file:

        for product in products:

            json_line = json.dumps(
                product,
                ensure_ascii=False,
            )

            file.write(
                json_line
                + "\n"
            )


# -------------------------------------------------------------
# Reservoir sampling
#
# WHY?
#
# We do NOT want to:
#
# 1. Load the complete 285 MB dataset into memory.
# 2. Simply take the first 800 products.
#
# Reservoir sampling lets us read the dataset as a stream while
# keeping only 800 randomly selected valid products in memory.
# -------------------------------------------------------------

def collect_products():

    rng = random.Random(
        RANDOM_SEED
    )

    sampled_products = []

    valid_count = 0
    raw_count = 0

    # Metadata should normally contain one row per parent ASIN,
    # but we still guard against duplicates.
    seen_product_ids = set()

    print(
        "Connecting to Amazon Reviews dataset..."
    )

    print(
        "Source:",
        DATA_URL
    )

    # ---------------------------------------------------------
    # stream=True:
    #
    # requests does not download the entire response into
    # memory before we start processing it.
    # ---------------------------------------------------------

    with requests.get(
        DATA_URL,
        stream=True,
        timeout=300,
    ) as response:

        response.raise_for_status()

        print(
            "Connected successfully."
        )

        print(
            "Streaming and curating products..."
        )

        # -----------------------------------------------------
        # Each physical line is one JSON product.
        # -----------------------------------------------------

        for line in response.iter_lines(
            decode_unicode=True
        ):

            if not line:
                continue

            raw_count += 1

            try:

                raw_product = json.loads(
                    line
                )

            except json.JSONDecodeError:

                continue

            curated = curate_product(
                raw_product
            )

            if curated is None:

                continue

            product_id = curated[
                "product_id"
            ]

            if product_id:

                if (
                    product_id
                    in seen_product_ids
                ):

                    continue

                seen_product_ids.add(
                    product_id
                )

            # -------------------------------------------------
            # Number of valid products encountered so far.
            # -------------------------------------------------

            valid_count += 1

            # -------------------------------------------------
            # First fill the reservoir with 800 products.
            # -------------------------------------------------

            if (
                len(sampled_products)
                < TOTAL_REQUIRED
            ):

                sampled_products.append(
                    curated
                )

            else:

                # ---------------------------------------------
                # Reservoir sampling:
                #
                # Choose a random position from all valid
                # examples seen so far.
                #
                # If it falls inside our reservoir size,
                # replace one existing sample.
                # ---------------------------------------------

                position = rng.randint(
                    1,
                    valid_count,
                )

                if (
                    position
                    <= TOTAL_REQUIRED
                ):

                    sampled_products[
                        position - 1
                    ] = curated

            if (
                raw_count % 50_000
                == 0
            ):

                print(
                    "Rows inspected:",
                    raw_count,
                    "| Valid products:",
                    valid_count,
                )

    print(
        "\nFinished reading dataset."
    )

    print(
        "Raw rows inspected:",
        raw_count,
    )

    print(
        "Valid priced products:",
        valid_count,
    )

    if (
        len(sampled_products)
        < TOTAL_REQUIRED
    ):

        raise RuntimeError(
            "Not enough valid products found. "
            f"Required {TOTAL_REQUIRED}, "
            f"found {len(sampled_products)}."
        )

    # ---------------------------------------------------------
    # Shuffle the final 800 selected products before splitting.
    # ---------------------------------------------------------

    rng.shuffle(
        sampled_products
    )

    return sampled_products


# -------------------------------------------------------------
# Main
# -------------------------------------------------------------

def main():

    products = collect_products()

    # ---------------------------------------------------------
    # Create splits
    # ---------------------------------------------------------

    train_end = TRAIN_SIZE

    validation_end = (
        TRAIN_SIZE
        + VALIDATION_SIZE
    )

    train_products = products[
        :train_end
    ]

    validation_products = products[
        train_end:
        validation_end
    ]

    test_products = products[
        validation_end:
    ]

    # ---------------------------------------------------------
    # Save splits
    # ---------------------------------------------------------

    write_jsonl(
        TRAIN_FILE,
        train_products,
    )

    write_jsonl(
        VALIDATION_FILE,
        validation_products,
    )

    write_jsonl(
        TEST_FILE,
        test_products,
    )

    # ---------------------------------------------------------
    # Summary
    # ---------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        "DATASET PREPARATION COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        "Training products:",
        len(
            train_products
        ),
    )

    print(
        "Validation products:",
        len(
            validation_products
        ),
    )

    print(
        "Test products:",
        len(
            test_products
        ),
    )

    print(
        "\nFILES"
    )

    print(
        "Train:",
        os.path.abspath(
            TRAIN_FILE
        )
    )

    print(
        "Validation:",
        os.path.abspath(
            VALIDATION_FILE
        )
    )

    print(
        "Test:",
        os.path.abspath(
            TEST_FILE
        )
    )

    # ---------------------------------------------------------
    # Print one example from each split.
    # ---------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TRAIN EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            train_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "VALIDATION EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            validation_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TEST EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            test_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )


if __name__ == "__main__":
    main()

Connecting to Amazon Reviews dataset...
Source: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_Appliances.jsonl
Connected successfully.
Streaming and curating products...

Finished reading dataset.
Raw rows inspected: 94327
Valid priced products: 45981

DATASET PREPARATION COMPLETE
Training products: 500
Validation products: 50
Test products: 250

FILES
Train: /content/amazon_price_train.jsonl
Validation: /content/amazon_price_validation.jsonl
Test: /content/amazon_price_test.jsonl

TRAIN EXAMPLE
{
  "description": "GE WE11X99 Htr&Hsg Asm. This is an O.E.M. Authorized part Fits with various GE brand models Oem part # WE11X99 This is an authorized aftermarket product. Fits with various GE brand models. Oem part # WE11X99 item is manufactured in United States.",
  "price": 182.56,
  "product_id": "B004XL0OIU"
}

VALIDATION EXAMPLE
{
  "description": "SkykS Door Knob Covers 2 Pcs Door Knob Protector Door Knob Covers Soft Round Door Kn

In [ ]:
import json

import torch

from datasets import load_dataset

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from trl import (
    SFTConfig,
    SFTTrainer,
)


# =============================================================
# CONFIGURATION
# =============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

TRAIN_FILE = "amazon_price_train.jsonl"

VALIDATION_FILE = (
    "amazon_price_validation.jsonl"
)

OUTPUT_DIR = (
    "./qwen-price-training"
)

FINAL_ADAPTER_DIR = (
    "./qwen-price-lora-final"
)

METRICS_FILE = (
    "qwen-price-training-metrics.json"
)


# -------------------------------------------------------------
# Training settings
# -------------------------------------------------------------

NUM_EPOCHS = 3

# Use only one physical example at a time.
# This keeps GPU memory usage conservative on T4.
TRAIN_BATCH_SIZE = 1

EVAL_BATCH_SIZE = 1

# 1 physical example × 4 accumulation steps
# = effective batch size 4
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 1e-4

MAX_LENGTH = 256


# -------------------------------------------------------------
# LoRA settings
# -------------------------------------------------------------

LORA_RANK = 16

LORA_ALPHA = 32

LORA_DROPOUT = 0.05


# =============================================================
# BUILD CHAT TRAINING EXAMPLE
# =============================================================

def build_messages(example):

    price = example["price"]

    return {
        "messages": [
            {
                "role": "system",
                "content":
                    "You estimate product prices. "
                    "Reply only with the price "
                    "in US dollars.",
            },
            {
                "role": "user",
                "content":
                    example["description"],
            },
            {
                "role": "assistant",
                "content":
                    f"${price:.2f}",
            },
        ]
    }


# =============================================================
# LOAD DATASETS
# =============================================================

def load_training_datasets():

    train_dataset = load_dataset(
        "json",
        data_files=TRAIN_FILE,
        split="train",
    )

    validation_dataset = load_dataset(
        "json",
        data_files=VALIDATION_FILE,
        split="train",
    )

    print(
        "Raw training examples:",
        len(train_dataset)
    )

    print(
        "Raw validation examples:",
        len(validation_dataset)
    )

    # ---------------------------------------------------------
    # Convert:
    #
    # description + price
    #
    # into:
    #
    # system/user/assistant messages
    # ---------------------------------------------------------

    train_dataset = train_dataset.map(
        build_messages,
        remove_columns=(
            train_dataset.column_names
        ),
    )

    validation_dataset = (
        validation_dataset.map(
            build_messages,
            remove_columns=(
                validation_dataset.column_names
            ),
        )
    )

    return (
        train_dataset,
        validation_dataset,
    )


# =============================================================
# TOKENIZER
# =============================================================

def load_tokenizer():

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_NAME
        )
    )

    # ---------------------------------------------------------
    # A pad token is required when batching examples.
    # ---------------------------------------------------------

    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    tokenizer.padding_side = "right"

    return tokenizer


# =============================================================
# LOAD 4-BIT MODEL
# =============================================================

def load_quantized_model():

    # ---------------------------------------------------------
    # IMPORTANT CHANGE
    #
    # Previous version:
    #
    # bnb_4bit_compute_dtype=torch.float16
    #
    # Trainer:
    #
    # fp16=True
    #
    # That activates AMP + GradScaler.
    #
    # Our T4/PyTorch environment then encounters an internal
    # BF16 gradient and GradScaler crashes.
    #
    # ---------------------------------------------------------
    #
    # New configuration:
    #
    # Weight storage:
    #
    #     4-bit NF4
    #
    # Compute:
    #
    #     FP32
    #
    # Mixed precision:
    #
    #     disabled
    #
    # ---------------------------------------------------------

    quantization_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_compute_dtype=(
                torch.float32
            ),

            bnb_4bit_use_double_quant=True,
        )
    )

    model = (
        AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,

            quantization_config=(
                quantization_config
            ),

            # One GPU:
            # put the model explicitly on GPU 0.
            device_map={
                "": 0
            },

            # Non-quantized floating modules also use FP32.
            dtype=torch.float32,
        )
    )

    # ---------------------------------------------------------
    # KV cache is useful during generation but not training.
    # ---------------------------------------------------------

    model.config.use_cache = False

    # ---------------------------------------------------------
    # Prepare quantized model for PEFT training.
    # ---------------------------------------------------------

    model = (
        prepare_model_for_kbit_training(
            model
        )
    )

    return model


# =============================================================
# ATTACH LoRA
# =============================================================

def add_lora(model):

    lora_config = LoraConfig(

        r=LORA_RANK,

        lora_alpha=LORA_ALPHA,

        lora_dropout=LORA_DROPOUT,

        bias="none",

        task_type="CAUSAL_LM",

        # QLoRA-style targeting:
        # add LoRA adapters to applicable linear layers.
        target_modules="all-linear",
    )

    model = get_peft_model(
        model,
        lora_config,
    )

    return model


# =============================================================
# FORCE TRAINABLE ADAPTERS TO FP32
# =============================================================

def cast_trainable_parameters_to_fp32(
    model
):

    parameter_count = 0

    for parameter in model.parameters():

        if parameter.requires_grad:

            parameter.data = (
                parameter.data.float()
            )

            parameter_count += (
                parameter.numel()
            )

    print(
        "Trainable parameters forced to FP32:",
        f"{parameter_count:,}",
    )


# =============================================================
# DTYPE AUDIT
# =============================================================

def print_trainable_dtypes(model):

    dtype_counts = {}

    for parameter in model.parameters():

        if not parameter.requires_grad:
            continue

        dtype_name = str(
            parameter.dtype
        )

        dtype_counts[
            dtype_name
        ] = (
            dtype_counts.get(
                dtype_name,
                0,
            )
            + parameter.numel()
        )

    print()

    print(
        "TRAINABLE PARAMETER DTYPES"
    )

    print(
        "=" * 70
    )

    for (
        dtype_name,
        count,
    ) in dtype_counts.items():

        print(
            dtype_name,
            ":",
            f"{count:,}",
        )


# =============================================================
# CREATE TRAINING CONFIGURATION
# =============================================================

def create_training_config():

    return SFTConfig(

        # -----------------------------------------------------
        # Output/checkpoints
        # -----------------------------------------------------

        output_dir=OUTPUT_DIR,

        # -----------------------------------------------------
        # Epochs
        # -----------------------------------------------------

        num_train_epochs=(
            NUM_EPOCHS
        ),

        # -----------------------------------------------------
        # Physical batch sizes
        # -----------------------------------------------------

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        # -----------------------------------------------------
        # Effective batch:
        #
        # 1 × 4 = 4
        # -----------------------------------------------------

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        # -----------------------------------------------------
        # Learning rate
        # -----------------------------------------------------

        learning_rate=(
            LEARNING_RATE
        ),

        # -----------------------------------------------------
        # CRITICAL FIX
        #
        # Disable BOTH AMP modes.
        #
        # Therefore:
        #
        # no autocast
        # no FP16 GradScaler
        # no BF16 mixed precision
        # -----------------------------------------------------

        fp16=False,

        bf16=False,

        # -----------------------------------------------------
        # Sequence length
        # -----------------------------------------------------

        max_length=(
            MAX_LENGTH
        ),

        # -----------------------------------------------------
        # Save memory by recomputing some activations.
        # -----------------------------------------------------

        gradient_checkpointing=True,

        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },

        use_cache=False,

        # -----------------------------------------------------
        # Train only against assistant output.
        # -----------------------------------------------------

        assistant_only_loss=True,

        # -----------------------------------------------------
        # Keep data flow simple for this exercise.
        # -----------------------------------------------------

        packing=False,

        # -----------------------------------------------------
        # Use standard AdamW.
        #
        # Avoid introducing another fused or 8-bit optimizer
        # while debugging dtype compatibility.
        # -----------------------------------------------------

        optim="adamw_torch",

        # -----------------------------------------------------
        # Logging
        # -----------------------------------------------------

        logging_steps=10,

        logging_first_step=True,

        # -----------------------------------------------------
        # Validation every epoch
        # -----------------------------------------------------

        eval_strategy="epoch",

        # -----------------------------------------------------
        # Checkpoint every epoch
        # -----------------------------------------------------

        save_strategy="epoch",

        save_total_limit=2,

        # -----------------------------------------------------
        # No external logger
        # -----------------------------------------------------

        report_to="none",

        seed=42,
    )


# =============================================================
# SAVE METRICS
# =============================================================

def save_metrics(
    initial_eval,
    training_metrics,
    final_eval,
):

    metrics = {
        "initial_validation":
            initial_eval,

        "training":
            training_metrics,

        "final_validation":
            final_eval,
    }

    with open(
        METRICS_FILE,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            metrics,
            file,
            indent=2,
        )


# =============================================================
# MAIN
# =============================================================

def main():

    # ---------------------------------------------------------
    # 1. GPU validation
    # ---------------------------------------------------------

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required."
        )

    gpu_name = (
        torch.cuda.get_device_name(0)
    )

    capability = (
        torch.cuda.get_device_capability(0)
    )

    print(
        "GPU:",
        gpu_name
    )

    print(
        "CUDA capability:",
        capability
    )

    # ---------------------------------------------------------
    # 2. Experiment settings
    # ---------------------------------------------------------

    effective_batch_size = (
        TRAIN_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    )

    print(
        "Effective batch size:",
        effective_batch_size
    )

    print(
        "Epochs:",
        NUM_EPOCHS
    )

    print(
        "Learning rate:",
        LEARNING_RATE
    )

    print(
        "Mixed precision:",
        "DISABLED"
    )

    print(
        "4-bit compute dtype:",
        torch.float32
    )

    # ---------------------------------------------------------
    # 3. Load dataset
    # ---------------------------------------------------------

    print(
        "\nLoading datasets..."
    )

    (
        train_dataset,
        validation_dataset,
    ) = load_training_datasets()

    print()

    print(
        "FIRST TRAINING EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        train_dataset[0]
    )

    # ---------------------------------------------------------
    # 4. Tokenizer
    # ---------------------------------------------------------

    print(
        "\nLoading tokenizer..."
    )

    tokenizer = (
        load_tokenizer()
    )

    # ---------------------------------------------------------
    # 5. Quantized base model
    # ---------------------------------------------------------

    print(
        "\nLoading Qwen in 4-bit..."
    )

    model = (
        load_quantized_model()
    )

    print(
        "Base model memory footprint:",
        round(
            model.get_memory_footprint()
            / 1024**3,
            2,
        ),
        "GB",
    )

    # ---------------------------------------------------------
    # 6. LoRA
    # ---------------------------------------------------------

    print(
        "\nAttaching LoRA..."
    )

    model = add_lora(
        model
    )

    print()

    print(
        "TRAINABLE PARAMETERS"
    )

    print(
        "=" * 70
    )

    model.print_trainable_parameters()

    # ---------------------------------------------------------
    # 7. Ensure adapters are FP32
    # ---------------------------------------------------------

    cast_trainable_parameters_to_fp32(
        model
    )

    print_trainable_dtypes(
        model
    )

    # ---------------------------------------------------------
    # 8. Training configuration
    # ---------------------------------------------------------

    training_config = (
        create_training_config()
    )

    # ---------------------------------------------------------
    # 9. Trainer
    # ---------------------------------------------------------

    trainer = SFTTrainer(

        model=model,

        args=training_config,

        train_dataset=train_dataset,

        eval_dataset=validation_dataset,

        processing_class=tokenizer,
    )

    # ---------------------------------------------------------
    # 10. Initial validation
    # ---------------------------------------------------------

    print()

    print(
        "INITIAL VALIDATION"
    )

    print(
        "=" * 70
    )

    initial_eval = (
        trainer.evaluate()
    )

    print(
        "Initial eval loss:",
        initial_eval.get(
            "eval_loss"
        )
    )

    # ---------------------------------------------------------
    # 11. Training
    # ---------------------------------------------------------

    print()

    print(
        "STARTING QLORA TRAINING"
    )

    print(
        "=" * 70
    )

    train_result = (
        trainer.train()
    )

    # ---------------------------------------------------------
    # 12. Final validation
    # ---------------------------------------------------------

    print()

    print(
        "FINAL VALIDATION"
    )

    print(
        "=" * 70
    )

    final_eval = (
        trainer.evaluate()
    )

    print(
        "Final eval loss:",
        final_eval.get(
            "eval_loss"
        )
    )

    # ---------------------------------------------------------
    # 13. Save adapter
    # ---------------------------------------------------------

    print(
        "\nSaving LoRA adapter..."
    )

    model.save_pretrained(
        FINAL_ADAPTER_DIR
    )

    tokenizer.save_pretrained(
        FINAL_ADAPTER_DIR
    )

    # ---------------------------------------------------------
    # 14. Save metrics
    # ---------------------------------------------------------

    save_metrics(
        initial_eval,
        train_result.metrics,
        final_eval,
    )

    # ---------------------------------------------------------
    # 15. Summary
    # ---------------------------------------------------------

    print()

    print(
        "=" * 70
    )

    print(
        "TRAINING COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        "Initial eval loss:",
        initial_eval.get(
            "eval_loss"
        )
    )

    print(
        "Final eval loss:",
        final_eval.get(
            "eval_loss"
        )
    )

    print(
        "Training loss:",
        train_result.metrics.get(
            "train_loss"
        )
    )

    print(
        "Adapter saved to:",
        FINAL_ADAPTER_DIR
    )

    print(
        "Metrics saved to:",
        METRICS_FILE
    )


if __name__ == "__main__":
    main()

GPU: Tesla T4
CUDA capability: (7, 5)
Effective batch size: 4
Epochs: 3
Learning rate: 0.0001
Mixed precision: DISABLED
4-bit compute dtype: torch.float32

Loading datasets...
Raw training examples: 500
Raw validation examples: 50

FIRST TRAINING EXAMPLE
{'messages': [{'content': 'You estimate product prices. Reply only with the price in US dollars.', 'role': 'system'}, {'content': 'GE WE11X99 Htr&Hsg Asm. This is an O.E.M. Authorized part Fits with various GE brand models Oem part # WE11X99 This is an authorized aftermarket product. Fits with various GE brand models. Oem part # WE11X99 item is manufactured in United States.', 'role': 'user'}, {'content': '$182.56', 'role': 'assistant'}]}

Loading tokenizer...

Loading Qwen in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model memory footprint: 1.48 GB

Attaching LoRA...

TRAINABLE PARAMETERS
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Trainable parameters forced to FP32: 18,464,768

TRAINABLE PARAMETER DTYPES
torch.float32 : 18,464,768

INITIAL VALIDATION


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
No log,3.629554,0,0.727090,0.000000,0.525422


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Initial eval loss: 3.629554271697998

STARTING QLORA TRAINING


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.940157,0.933643,0.986727,36060.000000,0.698500
2,0.830337,0.958093,0.830718,72120.000000,0.683966
3,0.843509,1.008743,0.769680,108180.000000,0.690997



FINAL VALIDATION


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.843509,1.008743,3,0.769680,108180.000000,0.690997


Final eval loss: 1.0087425708770752

Saving LoRA adapter...

TRAINING COMPLETE
Initial eval loss: 3.629554271697998
Final eval loss: 1.0087425708770752
Training loss: 0.8889970135414738
Adapter saved to: ./qwen-price-lora-final
Metrics saved to: qwen-price-training-metrics.json


In [ ]:
import csv
import json
import math
import os
import re

import torch

from peft import PeftModel

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# =============================================================
# CONFIGURATION
# =============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = "./qwen-price-lora-final"

TEST_FILE = "amazon_price_test.jsonl"

CSV_OUTPUT_FILE = (
    "qwen_price_evaluation_results.csv"
)

JSON_OUTPUT_FILE = (
    "qwen_price_evaluation_results.json"
)

SUMMARY_FILE = (
    "qwen_price_evaluation_summary.json"
)


# -------------------------------------------------------------
# Maximum generated tokens.
#
# We only expect something like:
#
# $129.99
#
# so we do not need a long response.
# -------------------------------------------------------------

MAX_NEW_TOKENS = 16


# =============================================================
# LOAD TEST DATA
# =============================================================

def load_test_products():

    products = []

    with open(
        TEST_FILE,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            product = json.loads(
                line
            )

            products.append(
                product
            )

    return products


# =============================================================
# LOAD TOKENIZER + MODEL + ADAPTER
# =============================================================

def load_model():

    if not os.path.isdir(
        ADAPTER_PATH
    ):

        raise FileNotFoundError(
            f"LoRA adapter not found: "
            f"{ADAPTER_PATH}\n"
            f"Finish script 70 first."
        )

    # ---------------------------------------------------------
    # Keep inference configuration close to our final
    # training configuration.
    #
    # Base weights:
    #     4-bit NF4
    #
    # Compute:
    #     FP32
    # ---------------------------------------------------------

    quantization_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_compute_dtype=(
                torch.float32
            ),

            bnb_4bit_use_double_quant=True,
        )
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_NAME
        )
    )

    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    # ---------------------------------------------------------
    # Load original Qwen.
    # ---------------------------------------------------------

    base_model = (
        AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,

            quantization_config=(
                quantization_config
            ),

            device_map={
                "": 0
            },

            dtype=torch.float32,
        )
    )

    # ---------------------------------------------------------
    # Attach the trained LoRA adapter.
    #
    # This creates:
    #
    # Base Qwen + trained LoRA
    # ---------------------------------------------------------

    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_PATH,
        is_trainable=False,
    )

    model.eval()

    return (
        model,
        tokenizer,
    )


# =============================================================
# BUILD MODEL INPUT
# =============================================================

def prepare_inputs(
    tokenizer,
    model,
    description,
):

    messages = [
        {
            "role": "system",
            "content":
                "You estimate product prices. "
                "Reply only with the price "
                "in US dollars.",
        },
        {
            "role": "user",
            "content":
                description,
        },
    ]

    inputs = (
        tokenizer.apply_chat_template(
            messages,

            tokenize=True,

            add_generation_prompt=True,

            return_tensors="pt",

            return_dict=True,
        )
    )

    # ---------------------------------------------------------
    # Put input tensors on the same GPU as the model.
    # ---------------------------------------------------------

    input_device = (
        model.get_input_embeddings()
        .weight
        .device
    )

    inputs = inputs.to(
        input_device
    )

    return inputs


# =============================================================
# GENERATE RESPONSE
# =============================================================

def generate_response(
    model,
    tokenizer,
    description,
):

    inputs = prepare_inputs(
        tokenizer,
        model,
        description,
    )

    input_length = (
        inputs["input_ids"].shape[1]
    )

    with torch.no_grad():

        output = model.generate(
            **inputs,

            max_new_tokens=(
                MAX_NEW_TOKENS
            ),

            # Deterministic generation.
            do_sample=False,

            pad_token_id=(
                tokenizer.eos_token_id
            ),
        )

    # ---------------------------------------------------------
    # generate() returns:
    #
    # original prompt tokens
    # +
    # newly generated tokens
    #
    # We only want the new tokens.
    # ---------------------------------------------------------

    generated_tokens = (
        output[0][
            input_length:
        ]
    )

    response = tokenizer.decode(
        generated_tokens,

        skip_special_tokens=True,
    )

    return response.strip()


# =============================================================
# EXTRACT NUMERIC PRICE
# =============================================================

def extract_price(response):

    # ---------------------------------------------------------
    # Supports:
    #
    # $199
    # 199
    # $199.99
    # $1,299.99
    #
    # ---------------------------------------------------------

    match = re.search(
        r"\$?\s*(\d[\d,]*(?:\.\d+)?)",
        response,
    )

    if match is None:

        return None

    number_text = (
        match
        .group(1)
        .replace(
            ",",
            "",
        )
    )

    try:

        price = float(
            number_text
        )

    except ValueError:

        return None

    return max(
        0.0,
        price,
    )


# =============================================================
# BASE MODEL PREDICTION
# =============================================================

def predict_base(
    model,
    tokenizer,
    description,
):

    # ---------------------------------------------------------
    # Temporarily disable LoRA.
    #
    # The model now behaves as original Base Qwen.
    # ---------------------------------------------------------

    with model.disable_adapter():

        response = generate_response(
            model,
            tokenizer,
            description,
        )

    price = extract_price(
        response
    )

    return (
        price,
        response,
    )


# =============================================================
# FINE-TUNED MODEL PREDICTION
# =============================================================

def predict_finetuned(
    model,
    tokenizer,
    description,
):

    # ---------------------------------------------------------
    # LoRA is enabled normally.
    # ---------------------------------------------------------

    response = generate_response(
        model,
        tokenizer,
        description,
    )

    price = extract_price(
        response
    )

    return (
        price,
        response,
    )


# =============================================================
# MAE
# =============================================================

def calculate_mae(
    actual_values,
    predicted_values,
):

    total_error = 0.0

    for (
        actual,
        predicted,
    ) in zip(
        actual_values,
        predicted_values,
    ):

        total_error += abs(
            actual - predicted
        )

    return (
        total_error
        / len(actual_values)
    )


# =============================================================
# RMSLE
# =============================================================

def calculate_rmsle(
    actual_values,
    predicted_values,
):

    squared_log_errors = []

    for (
        actual,
        predicted,
    ) in zip(
        actual_values,
        predicted_values,
    ):

        log_actual = math.log1p(
            actual
        )

        log_predicted = math.log1p(
            max(
                0.0,
                predicted,
            )
        )

        error = (
            log_predicted
            - log_actual
        )

        squared_log_errors.append(
            error ** 2
        )

    mean_squared_log_error = (
        sum(
            squared_log_errors
        )
        / len(
            squared_log_errors
        )
    )

    return math.sqrt(
        mean_squared_log_error
    )


# =============================================================
# SAVE CSV
# =============================================================

def save_csv(results):

    fieldnames = [
        "product_id",

        "actual_price",

        "base_price",

        "fine_tuned_price",

        "base_absolute_error",

        "fine_tuned_absolute_error",

        "base_response",

        "fine_tuned_response",

        "description",
    ]

    with open(
        CSV_OUTPUT_FILE,
        "w",
        encoding="utf-8",
        newline="",
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        writer.writerows(
            results
        )


# =============================================================
# SAVE JSON
# =============================================================

def save_json(results):

    with open(
        JSON_OUTPUT_FILE,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            results,
            file,
            indent=2,
            ensure_ascii=False,
        )


# =============================================================
# MAIN
# =============================================================

def main():

    # ---------------------------------------------------------
    # 1. Check GPU
    # ---------------------------------------------------------

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required."
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    # ---------------------------------------------------------
    # 2. Load untouched test dataset
    # ---------------------------------------------------------

    print(
        "\nLoading test products..."
    )

    test_products = (
        load_test_products()
    )

    print(
        "Test products:",
        len(
            test_products
        )
    )

    # ---------------------------------------------------------
    # 3. Load Qwen + LoRA
    # ---------------------------------------------------------

    print(
        "\nLoading Qwen + trained LoRA..."
    )

    (
        model,
        tokenizer,
    ) = load_model()

    print(
        "Model loaded."
    )

    # ---------------------------------------------------------
    # 4. Containers for metrics/results
    # ---------------------------------------------------------

    results = []

    actual_prices = []

    base_predictions = []

    fine_tuned_predictions = []

    base_parse_failures = 0

    fine_tuned_parse_failures = 0

    # ---------------------------------------------------------
    # 5. Evaluate every product
    # ---------------------------------------------------------

    for (
        index,
        product,
    ) in enumerate(
        test_products,
        start=1,
    ):

        description = (
            product[
                "description"
            ]
        )

        actual_price = float(
            product[
                "price"
            ]
        )

        product_id = (
            product.get(
                "product_id",
                "",
            )
        )

        # -----------------------------------------------------
        # Base Qwen
        # -----------------------------------------------------

        (
            base_price,
            base_response,
        ) = predict_base(
            model,
            tokenizer,
            description,
        )

        # -----------------------------------------------------
        # Fine-tuned Qwen
        # -----------------------------------------------------

        (
            fine_tuned_price,
            fine_tuned_response,
        ) = predict_finetuned(
            model,
            tokenizer,
            description,
        )

        # -----------------------------------------------------
        # Handle unparsable output.
        #
        # We count it separately.
        #
        # For metrics we use 0.0 so failures are penalized
        # rather than silently ignored.
        # -----------------------------------------------------

        if base_price is None:

            base_parse_failures += 1

            base_price = 0.0

        if fine_tuned_price is None:

            fine_tuned_parse_failures += 1

            fine_tuned_price = 0.0

        # -----------------------------------------------------
        # Errors
        # -----------------------------------------------------

        base_error = abs(
            actual_price
            - base_price
        )

        fine_tuned_error = abs(
            actual_price
            - fine_tuned_price
        )

        actual_prices.append(
            actual_price
        )

        base_predictions.append(
            base_price
        )

        fine_tuned_predictions.append(
            fine_tuned_price
        )

        results.append(
            {
                "product_id":
                    product_id,

                "actual_price":
                    actual_price,

                "base_price":
                    base_price,

                "fine_tuned_price":
                    fine_tuned_price,

                "base_absolute_error":
                    round(
                        base_error,
                        2,
                    ),

                "fine_tuned_absolute_error":
                    round(
                        fine_tuned_error,
                        2,
                    ),

                "base_response":
                    base_response,

                "fine_tuned_response":
                    fine_tuned_response,

                "description":
                    description,
            }
        )

        # -----------------------------------------------------
        # Progress display
        # -----------------------------------------------------

        print(
            f"[{index}/{len(test_products)}] "
            f"Actual=${actual_price:.2f} | "
            f"Base=${base_price:.2f} | "
            f"Fine-tuned=${fine_tuned_price:.2f}"
        )

    # ---------------------------------------------------------
    # 6. Calculate Base metrics
    # ---------------------------------------------------------

    base_mae = calculate_mae(
        actual_prices,
        base_predictions,
    )

    base_rmsle = calculate_rmsle(
        actual_prices,
        base_predictions,
    )

    # ---------------------------------------------------------
    # 7. Calculate Fine-tuned metrics
    # ---------------------------------------------------------

    fine_tuned_mae = calculate_mae(
        actual_prices,
        fine_tuned_predictions,
    )

    fine_tuned_rmsle = (
        calculate_rmsle(
            actual_prices,
            fine_tuned_predictions,
        )
    )

    # ---------------------------------------------------------
    # 8. Save per-product results
    # ---------------------------------------------------------

    save_csv(
        results
    )

    save_json(
        results
    )

    # ---------------------------------------------------------
    # 9. Build summary
    # ---------------------------------------------------------

    summary = {

        "test_products":
            len(
                test_products
            ),

        "base": {

            "mae":
                base_mae,

            "rmsle":
                base_rmsle,

            "parse_failures":
                base_parse_failures,
        },

        "fine_tuned": {

            "mae":
                fine_tuned_mae,

            "rmsle":
                fine_tuned_rmsle,

            "parse_failures":
                fine_tuned_parse_failures,
        },

        "mae_improvement": (
            base_mae
            - fine_tuned_mae
        ),

        "rmsle_improvement": (
            base_rmsle
            - fine_tuned_rmsle
        ),
    }

    # ---------------------------------------------------------
    # 10. Save summary
    # ---------------------------------------------------------

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            summary,
            file,
            indent=2,
        )

    # ---------------------------------------------------------
    # 11. Print final comparison
    # ---------------------------------------------------------

    print()

    print(
        "=" * 70
    )

    print(
        "FINAL HELD-OUT TEST RESULTS"
    )

    print(
        "=" * 70
    )

    print(
        "Products:",
        len(
            test_products
        )
    )

    print()

    print(
        "BASE QWEN"
    )

    print(
        "MAE:",
        round(
            base_mae,
            2,
        )
    )

    print(
        "RMSLE:",
        round(
            base_rmsle,
            4,
        )
    )

    print(
        "Parse failures:",
        base_parse_failures,
    )

    print()

    print(
        "FINE-TUNED QWEN"
    )

    print(
        "MAE:",
        round(
            fine_tuned_mae,
            2,
        )
    )

    print(
        "RMSLE:",
        round(
            fine_tuned_rmsle,
            4,
        )
    )

    print(
        "Parse failures:",
        fine_tuned_parse_failures,
    )

    print()

    print(
        "MAE improvement:",
        round(
            base_mae
            - fine_tuned_mae,
            2,
        )
    )

    print(
        "RMSLE improvement:",
        round(
            base_rmsle
            - fine_tuned_rmsle,
            4,
        )
    )

    print()

    print(
        "Detailed CSV:",
        CSV_OUTPUT_FILE
    )

    print(
        "Detailed JSON:",
        JSON_OUTPUT_FILE
    )

    print(
        "Summary:",
        SUMMARY_FILE
    )


if __name__ == "__main__":
    main()

GPU: Tesla T4

Loading test products...
Test products: 250

Loading Qwen + trained LoRA...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded.
[1/250] Actual=$27.99 | Base=$1200.00 | Fine-tuned=$50.99
[2/250] Actual=$67.30 | Base=$29.99 | Fine-tuned=$97.28
[3/250] Actual=$242.14 | Base=$15.00 | Fine-tuned=$20.99
[4/250] Actual=$14.99 | Base=$29.99 | Fine-tuned=$12.99
[5/250] Actual=$30.99 | Base=$7.99 | Fine-tuned=$28.99
[6/250] Actual=$22.98 | Base=$129.99 | Fine-tuned=$32.99
[7/250] Actual=$26.27 | Base=$15.99 | Fine-tuned=$20.99
[8/250] Actual=$105.22 | Base=$20.00 | Fine-tuned=$20.38
[9/250] Actual=$51.95 | Base=$34.99 | Fine-tuned=$34.99
[10/250] Actual=$9.41 | Base=$19.99 | Fine-tuned=$24.99
[11/250] Actual=$29.58 | Base=$19.99 | Fine-tuned=$30.99
[12/250] Actual=$33.99 | Base=$5.00 | Fine-tuned=$50.99
[13/250] Actual=$25.99 | Base=$99.99 | Fine-tuned=$79.99
[14/250] Actual=$29.99 | Base=$49.99 | Fine-tuned=$14.99
[15/250] Actual=$12.89 | Base=$5.99 | Fine-tuned=$9.99
[16/250] Actual=$19.99 | Base=$19.99 | Fine-tuned=$20.99
[17/250] Actual=$12.54 | Base=$29.99 | Fine-tuned=$19.99
[18/250] Actual=$9.75 | Ba

In [ ]:
import csv
import gc
import json
import os

import torch

from datasets import load_dataset

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from trl import (
    SFTConfig,
    SFTTrainer,
)


# =============================================================
# CONFIGURATION
# =============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

TRAIN_FILE = "amazon_price_train.jsonl"

VALIDATION_FILE = (
    "amazon_price_validation.jsonl"
)


EXPERIMENT_ROOT = (
    "./lora_rank_experiments"
)

CSV_RESULT_FILE = (
    "lora_rank_experiment_results.csv"
)

JSON_RESULT_FILE = (
    "lora_rank_experiment_results.json"
)


# -------------------------------------------------------------
# Controlled rank experiments
#
# alpha / rank = 2 for every experiment
# -------------------------------------------------------------

LORA_EXPERIMENTS = [
    {
        "rank": 4,
        "alpha": 8,
    },
    {
        "rank": 16,
        "alpha": 32,
    },
    {
        "rank": 64,
        "alpha": 128,
    },
]


# -------------------------------------------------------------
# Everything below stays constant.
# -------------------------------------------------------------

LORA_DROPOUT = 0.05

NUM_EPOCHS = 3

TRAIN_BATCH_SIZE = 1

EVAL_BATCH_SIZE = 1

GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 1e-4

MAX_LENGTH = 256

RANDOM_SEED = 42


# =============================================================
# DATASET
# =============================================================

def build_messages(example):

    price = example["price"]

    return {
        "messages": [
            {
                "role": "system",
                "content":
                    "You estimate product prices. "
                    "Reply only with the price "
                    "in US dollars.",
            },
            {
                "role": "user",
                "content":
                    example["description"],
            },
            {
                "role": "assistant",
                "content":
                    f"${price:.2f}",
            },
        ]
    }


def load_training_datasets():

    train_dataset = load_dataset(
        "json",
        data_files=TRAIN_FILE,
        split="train",
    )

    validation_dataset = load_dataset(
        "json",
        data_files=VALIDATION_FILE,
        split="train",
    )

    train_dataset = train_dataset.map(
        build_messages,
        remove_columns=(
            train_dataset.column_names
        ),
    )

    validation_dataset = (
        validation_dataset.map(
            build_messages,
            remove_columns=(
                validation_dataset.column_names
            ),
        )
    )

    return (
        train_dataset,
        validation_dataset,
    )


# =============================================================
# TOKENIZER
# =============================================================

def load_tokenizer():

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_NAME
        )
    )

    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    tokenizer.padding_side = "right"

    return tokenizer


# =============================================================
# LOAD FRESH 4-BIT BASE MODEL
# =============================================================

def load_base_model():

    quantization_config = (
        BitsAndBytesConfig(

            load_in_4bit=True,

            bnb_4bit_quant_type="nf4",

            # Stable configuration from our T4 experiment
            bnb_4bit_compute_dtype=(
                torch.float32
            ),

            bnb_4bit_use_double_quant=True,
        )
    )

    model = (
        AutoModelForCausalLM.from_pretrained(

            MODEL_NAME,

            quantization_config=(
                quantization_config
            ),

            device_map={
                "": 0
            },

            dtype=torch.float32,
        )
    )

    model.config.use_cache = False

    model = (
        prepare_model_for_kbit_training(
            model
        )
    )

    return model


# =============================================================
# ADD LoRA
# =============================================================

def add_lora(
    model,
    rank,
    alpha,
):

    lora_config = LoraConfig(

        r=rank,

        lora_alpha=alpha,

        lora_dropout=(
            LORA_DROPOUT
        ),

        bias="none",

        task_type="CAUSAL_LM",

        # QLoRA-style:
        # add adapters to all linear layers
        target_modules="all-linear",
    )

    model = get_peft_model(
        model,
        lora_config,
    )

    return model


# =============================================================
# FORCE TRAINABLE PARAMETERS TO FP32
# =============================================================

def cast_trainable_parameters_to_fp32(
    model
):

    for parameter in model.parameters():

        if parameter.requires_grad:

            parameter.data = (
                parameter.data.float()
            )


# =============================================================
# COUNT TRAINABLE PARAMETERS
# =============================================================

def count_trainable_parameters(
    model
):

    trainable = 0

    total = 0

    for parameter in model.parameters():

        total += parameter.numel()

        if parameter.requires_grad:

            trainable += (
                parameter.numel()
            )

    percentage = (
        trainable
        / total
        * 100
    )

    return (
        trainable,
        total,
        percentage,
    )


# =============================================================
# TRAINING CONFIG
# =============================================================

def create_training_config(
    output_dir
):

    return SFTConfig(

        output_dir=output_dir,

        num_train_epochs=(
            NUM_EPOCHS
        ),

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        learning_rate=(
            LEARNING_RATE
        ),

        # -----------------------------------------------------
        # Stable configuration for our T4 runtime
        # -----------------------------------------------------

        fp16=False,

        bf16=False,

        optim="adamw_torch",

        max_length=(
            MAX_LENGTH
        ),

        gradient_checkpointing=True,

        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },

        use_cache=False,

        assistant_only_loss=True,

        packing=False,

        logging_steps=10,

        logging_first_step=True,

        # -----------------------------------------------------
        # Validation-based model selection
        # -----------------------------------------------------

        eval_strategy="epoch",

        save_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model=(
            "eval_loss"
        ),

        greater_is_better=False,

        save_total_limit=3,

        report_to="none",

        seed=RANDOM_SEED,
    )


# =============================================================
# CALCULATE SAVED DIRECTORY SIZE
# =============================================================

def get_directory_size_mb(
    directory
):

    total_bytes = 0

    for (
        root,
        directories,
        files,
    ) in os.walk(
        directory
    ):

        for filename in files:

            file_path = os.path.join(
                root,
                filename,
            )

            total_bytes += (
                os.path.getsize(
                    file_path
                )
            )

    return (
        total_bytes
        / 1024
        / 1024
    )


# =============================================================
# RELEASE GPU MEMORY
# =============================================================

def cleanup_gpu():

    gc.collect()

    torch.cuda.empty_cache()


# =============================================================
# RUN ONE RANK EXPERIMENT
# =============================================================

def run_experiment(
    rank,
    alpha,
    tokenizer,
    train_dataset,
    validation_dataset,
):

    print()

    print(
        "=" * 80
    )

    print(
        f"LoRA EXPERIMENT: "
        f"rank={rank}, "
        f"alpha={alpha}"
    )

    print(
        "=" * 80
    )

    experiment_dir = os.path.join(
        EXPERIMENT_ROOT,
        f"rank_{rank}",
    )

    checkpoint_dir = os.path.join(
        experiment_dir,
        "checkpoints",
    )

    adapter_dir = os.path.join(
        experiment_dir,
        "best_adapter",
    )

    # ---------------------------------------------------------
    # IMPORTANT:
    #
    # Each experiment starts from a completely fresh Qwen
    # base model.
    #
    # Otherwise rank=16 would inherit training from rank=4,
    # which would make the comparison invalid.
    # ---------------------------------------------------------

    print(
        "\nLoading fresh Qwen base model..."
    )

    model = load_base_model()

    print(
        "Attaching LoRA..."
    )

    model = add_lora(
        model,
        rank,
        alpha,
    )

    cast_trainable_parameters_to_fp32(
        model
    )

    (
        trainable_parameters,
        total_parameters,
        trainable_percentage,
    ) = count_trainable_parameters(
        model
    )

    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}"
    )

    print(
        "Trainable percentage:",
        round(
            trainable_percentage,
            4,
        ),
        "%",
    )

    # ---------------------------------------------------------
    # Trainer
    # ---------------------------------------------------------

    training_config = (
        create_training_config(
            checkpoint_dir
        )
    )

    trainer = SFTTrainer(

        model=model,

        args=training_config,

        train_dataset=train_dataset,

        eval_dataset=validation_dataset,

        processing_class=tokenizer,
    )

    # ---------------------------------------------------------
    # Validation before training
    # ---------------------------------------------------------

    print(
        "\nInitial validation..."
    )

    initial_eval = (
        trainer.evaluate()
    )

    initial_validation_loss = (
        initial_eval[
            "eval_loss"
        ]
    )

    print(
        "Initial validation loss:",
        initial_validation_loss,
    )

    # ---------------------------------------------------------
    # Train
    # ---------------------------------------------------------

    print(
        "\nTraining..."
    )

    train_result = (
        trainer.train()
    )

    # ---------------------------------------------------------
    # Trainer has reloaded the best validation checkpoint.
    # ---------------------------------------------------------

    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )

    best_validation_loss = (
        trainer.state.best_metric
    )

    print(
        "\nBest checkpoint:",
        best_checkpoint,
    )

    print(
        "Best validation loss:",
        best_validation_loss,
    )

    # ---------------------------------------------------------
    # Validate the reloaded best adapter
    # ---------------------------------------------------------

    best_eval = (
        trainer.evaluate()
    )

    loaded_validation_loss = (
        best_eval[
            "eval_loss"
        ]
    )

    # ---------------------------------------------------------
    # Save best adapter
    # ---------------------------------------------------------

    trainer.model.save_pretrained(
        adapter_dir
    )

    tokenizer.save_pretrained(
        adapter_dir
    )

    adapter_size_mb = (
        get_directory_size_mb(
            adapter_dir
        )
    )

    # ---------------------------------------------------------
    # Training metrics
    # ---------------------------------------------------------

    training_loss = (
        train_result.metrics.get(
            "train_loss"
        )
    )

    training_runtime = (
        train_result.metrics.get(
            "train_runtime"
        )
    )

    result = {

        "rank":
            rank,

        "alpha":
            alpha,

        "alpha_to_rank_ratio":
            alpha / rank,

        "trainable_parameters":
            trainable_parameters,

        "total_parameters":
            total_parameters,

        "trainable_percentage":
            trainable_percentage,

        "initial_validation_loss":
            initial_validation_loss,

        "best_validation_loss":
            best_validation_loss,

        "loaded_best_validation_loss":
            loaded_validation_loss,

        "training_loss":
            training_loss,

        "training_runtime_seconds":
            training_runtime,

        "adapter_size_mb":
            adapter_size_mb,

        "best_checkpoint":
            best_checkpoint,

        "adapter_path":
            adapter_dir,
    }

    print()

    print(
        "EXPERIMENT RESULT"
    )

    print(
        "-" * 80
    )

    print(
        "Rank:",
        rank
    )

    print(
        "Alpha:",
        alpha
    )

    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}"
    )

    print(
        "Best validation loss:",
        round(
            best_validation_loss,
            6,
        )
    )

    print(
        "Adapter size:",
        round(
            adapter_size_mb,
            2,
        ),
        "MB",
    )

    # ---------------------------------------------------------
    # Delete model/trainer before starting next experiment.
    # ---------------------------------------------------------

    del trainer
    del model

    cleanup_gpu()

    return result


# =============================================================
# SAVE RESULTS
# =============================================================

def save_results(
    results
):

    with open(
        JSON_RESULT_FILE,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            results,
            file,
            indent=2,
        )

    fieldnames = [
        "rank",
        "alpha",
        "alpha_to_rank_ratio",
        "trainable_parameters",
        "total_parameters",
        "trainable_percentage",
        "initial_validation_loss",
        "best_validation_loss",
        "loaded_best_validation_loss",
        "training_loss",
        "training_runtime_seconds",
        "adapter_size_mb",
        "best_checkpoint",
        "adapter_path",
    ]

    with open(
        CSV_RESULT_FILE,
        "w",
        encoding="utf-8",
        newline="",
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        writer.writerows(
            results
        )


# =============================================================
# PRINT FINAL COMPARISON
# =============================================================

def print_comparison(
    results
):

    print()

    print(
        "=" * 100
    )

    print(
        "FINAL LoRA RANK COMPARISON"
    )

    print(
        "=" * 100
    )

    print(
        f"{'Rank':<8}"
        f"{'Alpha':<10}"
        f"{'Trainable Params':<22}"
        f"{'Best Val Loss':<20}"
        f"{'Adapter MB':<15}"
    )

    print(
        "-" * 100
    )

    for result in results:

        print(
            f"{result['rank']:<8}"
            f"{result['alpha']:<10}"
            f"{result['trainable_parameters']:<22,}"
            f"{result['best_validation_loss']:<20.6f}"
            f"{result['adapter_size_mb']:<15.2f}"
        )

    # ---------------------------------------------------------
    # Select by validation loss ONLY.
    #
    # We are deliberately not touching the held-out test set.
    # ---------------------------------------------------------

    best_result = min(
        results,
        key=lambda item:
            item[
                "best_validation_loss"
            ],
    )

    print()

    print(
        "Lowest validation loss:"
    )

    print(
        "Rank:",
        best_result[
            "rank"
        ]
    )

    print(
        "Alpha:",
        best_result[
            "alpha"
        ]
    )

    print(
        "Validation loss:",
        round(
            best_result[
                "best_validation_loss"
            ],
            6,
        )
    )

    print()

    print(
        "Results saved to:"
    )

    print(
        CSV_RESULT_FILE
    )

    print(
        JSON_RESULT_FILE
    )


# =============================================================
# MAIN
# =============================================================

def main():

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required."
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA capability:",
        torch.cuda.get_device_capability(
            0
        )
    )

    print(
        "\nLoading shared datasets..."
    )

    (
        train_dataset,
        validation_dataset,
    ) = load_training_datasets()

    print(
        "Training examples:",
        len(
            train_dataset
        )
    )

    print(
        "Validation examples:",
        len(
            validation_dataset
        )
    )

    print(
        "\nLoading tokenizer..."
    )

    tokenizer = load_tokenizer()

    os.makedirs(
        EXPERIMENT_ROOT,
        exist_ok=True,
    )

    results = []

    # ---------------------------------------------------------
    # Run:
    #
    # rank 4
    # rank 16
    # rank 64
    #
    # sequentially.
    # ---------------------------------------------------------

    for experiment in (
        LORA_EXPERIMENTS
    ):

        result = run_experiment(

            rank=experiment[
                "rank"
            ],

            alpha=experiment[
                "alpha"
            ],

            tokenizer=tokenizer,

            train_dataset=(
                train_dataset
            ),

            validation_dataset=(
                validation_dataset
            ),
        )

        results.append(
            result
        )

    save_results(
        results
    )

    print_comparison(
        results
    )


if __name__ == "__main__":
    main()

GPU: Tesla T4
CUDA capability: (7, 5)

Loading shared datasets...
Training examples: 500
Validation examples: 50

Loading tokenizer...

LoRA EXPERIMENT: rank=4, alpha=8

Loading fresh Qwen base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Attaching LoRA...
Trainable parameters: 4,616,192
Trainable percentage: 0.5168 %

Initial validation...


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
No log,3.629554,0,0.727090,0.000000,0.525422


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

Initial validation loss: 3.629554271697998

Training...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.942093,0.927729,1.012679,36060.000000,0.698065
2,0.797919,0.972522,0.766180,72120.000000,0.696156
3,0.806129,1.037867,0.716215,108180.000000,0.674157



Best checkpoint: ./lora_rank_experiments/rank_64/checkpoints/checkpoint-58
Best validation loss: 0.9277288317680359


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.806129,0.927735,3,1.012696,108180.000000,0.698065



EXPERIMENT RESULT
--------------------------------------------------------------------------------
Rank: 64
Alpha: 128
Trainable parameters: 73,859,072
Best validation loss: 0.927729
Adapter size: 292.7 MB

FINAL LoRA RANK COMPARISON
Rank    Alpha     Trainable Params      Best Val Loss       Adapter MB     
----------------------------------------------------------------------------------------------------
4       8         4,616,192             0.940207            28.56          
16      32        18,464,768            0.931580            81.39          
64      128       73,859,072            0.927729            292.70         

Lowest validation loss:
Rank: 64
Alpha: 128
Validation loss: 0.927729

Results saved to:
lora_rank_experiment_results.csv
lora_rank_experiment_results.json


In [ ]:
import json
import os
import random
import re

import requests


# -------------------------------------------------------------
# Configuration
# -------------------------------------------------------------

DATA_URL = (
    "https://huggingface.co/datasets/"
    "McAuley-Lab/Amazon-Reviews-2023/"
    "resolve/main/raw/meta_categories/"
    "meta_Appliances.jsonl"
)


TRAIN_SIZE = 500
VALIDATION_SIZE = 50
TEST_SIZE = 250


TOTAL_REQUIRED = (
    TRAIN_SIZE
    + VALIDATION_SIZE
    + TEST_SIZE
)


MIN_PRICE = 0.50
MAX_PRICE = 999.49


MAX_DESCRIPTION_CHARS = 2500


RANDOM_SEED = 42


TRAIN_FILE = "amazon_price_train.jsonl"

VALIDATION_FILE = (
    "amazon_price_validation.jsonl"
)

TEST_FILE = "amazon_price_test.jsonl"


# -------------------------------------------------------------
# Parse price
# -------------------------------------------------------------

def parse_price(value):

    if value is None:
        return None

    text = str(value).strip()

    if not text:
        return None

    text = text.replace(
        "$",
        "",
    )

    text = text.replace(
        ",",
        "",
    )

    match = re.search(
        r"\d+(?:\.\d+)?",
        text,
    )

    if not match:
        return None

    try:

        return float(
            match.group()
        )

    except ValueError:

        return None


# -------------------------------------------------------------
# Clean text
# -------------------------------------------------------------

def clean_text(value):

    if value is None:
        return ""

    text = str(value)

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


# -------------------------------------------------------------
# Convert list/string into normal text
# -------------------------------------------------------------

def join_text(value):

    if value is None:
        return ""

    if isinstance(
        value,
        list,
    ):

        cleaned_items = []

        for item in value:

            cleaned = clean_text(
                item
            )

            if cleaned:

                cleaned_items.append(
                    cleaned
                )

        return " ".join(
            cleaned_items
        )

    return clean_text(
        value
    )


# -------------------------------------------------------------
# Build product text
#
# We use:
#
# title
# +
# features
# +
# description
# -------------------------------------------------------------

def build_product_text(product):

    title = clean_text(
        product.get(
            "title"
        )
    )

    features = join_text(
        product.get(
            "features"
        )
    )

    description = join_text(
        product.get(
            "description"
        )
    )

    parts = []

    if title:

        parts.append(
            title
        )

    if features:

        parts.append(
            features
        )

    if description:

        parts.append(
            description
        )

    combined = " ".join(
        parts
    )

    combined = clean_text(
        combined
    )

    # ---------------------------------------------------------
    # Character-level truncation.
    #
    # Token-level truncation happens later during SFT.
    # ---------------------------------------------------------

    return combined[
        :MAX_DESCRIPTION_CHARS
    ]


# -------------------------------------------------------------
# Curate one Amazon product
# -------------------------------------------------------------

def curate_product(product):

    price = parse_price(
        product.get(
            "price"
        )
    )

    if price is None:

        return None

    if not (
        MIN_PRICE
        <= price
        <= MAX_PRICE
    ):

        return None

    description = build_product_text(
        product
    )

    # Very short descriptions contain little useful signal.
    if len(description) < 40:

        return None

    product_id = product.get(
        "parent_asin"
    )

    if product_id is None:

        product_id = ""

    return {
        "description": description,

        "price": round(
            price,
            2,
        ),

        "product_id": str(
            product_id
        ),
    }


# -------------------------------------------------------------
# Write JSONL
# -------------------------------------------------------------

def write_jsonl(
    filename,
    products,
):

    with open(
        filename,
        "w",
        encoding="utf-8",
    ) as file:

        for product in products:

            json_line = json.dumps(
                product,
                ensure_ascii=False,
            )

            file.write(
                json_line
                + "\n"
            )


# -------------------------------------------------------------
# Reservoir sampling
#
# WHY?
#
# We do NOT want to:
#
# 1. Load the complete 285 MB dataset into memory.
# 2. Simply take the first 800 products.
#
# Reservoir sampling lets us read the dataset as a stream while
# keeping only 800 randomly selected valid products in memory.
# -------------------------------------------------------------

def collect_products():

    rng = random.Random(
        RANDOM_SEED
    )

    sampled_products = []

    valid_count = 0
    raw_count = 0

    # Metadata should normally contain one row per parent ASIN,
    # but we still guard against duplicates.
    seen_product_ids = set()

    print(
        "Connecting to Amazon Reviews dataset..."
    )

    print(
        "Source:",
        DATA_URL
    )

    # ---------------------------------------------------------
    # stream=True:
    #
    # requests does not download the entire response into
    # memory before we start processing it.
    # ---------------------------------------------------------

    with requests.get(
        DATA_URL,
        stream=True,
        timeout=300,
    ) as response:

        response.raise_for_status()

        print(
            "Connected successfully."
        )

        print(
            "Streaming and curating products..."
        )

        # -----------------------------------------------------
        # Each physical line is one JSON product.
        # -----------------------------------------------------

        for line in response.iter_lines(
            decode_unicode=True
        ):

            if not line:
                continue

            raw_count += 1

            try:

                raw_product = json.loads(
                    line
                )

            except json.JSONDecodeError:

                continue

            curated = curate_product(
                raw_product
            )

            if curated is None:

                continue

            product_id = curated[
                "product_id"
            ]

            if product_id:

                if (
                    product_id
                    in seen_product_ids
                ):

                    continue

                seen_product_ids.add(
                    product_id
                )

            # -------------------------------------------------
            # Number of valid products encountered so far.
            # -------------------------------------------------

            valid_count += 1

            # -------------------------------------------------
            # First fill the reservoir with 800 products.
            # -------------------------------------------------

            if (
                len(sampled_products)
                < TOTAL_REQUIRED
            ):

                sampled_products.append(
                    curated
                )

            else:

                # ---------------------------------------------
                # Reservoir sampling:
                #
                # Choose a random position from all valid
                # examples seen so far.
                #
                # If it falls inside our reservoir size,
                # replace one existing sample.
                # ---------------------------------------------

                position = rng.randint(
                    1,
                    valid_count,
                )

                if (
                    position
                    <= TOTAL_REQUIRED
                ):

                    sampled_products[
                        position - 1
                    ] = curated

            if (
                raw_count % 50_000
                == 0
            ):

                print(
                    "Rows inspected:",
                    raw_count,
                    "| Valid products:",
                    valid_count,
                )

    print(
        "\nFinished reading dataset."
    )

    print(
        "Raw rows inspected:",
        raw_count,
    )

    print(
        "Valid priced products:",
        valid_count,
    )

    if (
        len(sampled_products)
        < TOTAL_REQUIRED
    ):

        raise RuntimeError(
            "Not enough valid products found. "
            f"Required {TOTAL_REQUIRED}, "
            f"found {len(sampled_products)}."
        )

    # ---------------------------------------------------------
    # Shuffle the final 800 selected products before splitting.
    # ---------------------------------------------------------

    rng.shuffle(
        sampled_products
    )

    return sampled_products


# -------------------------------------------------------------
# Main
# -------------------------------------------------------------

def main():

    products = collect_products()

    # ---------------------------------------------------------
    # Create splits
    # ---------------------------------------------------------

    train_end = TRAIN_SIZE

    validation_end = (
        TRAIN_SIZE
        + VALIDATION_SIZE
    )

    train_products = products[
        :train_end
    ]

    validation_products = products[
        train_end:
        validation_end
    ]

    test_products = products[
        validation_end:
    ]

    # ---------------------------------------------------------
    # Save splits
    # ---------------------------------------------------------

    write_jsonl(
        TRAIN_FILE,
        train_products,
    )

    write_jsonl(
        VALIDATION_FILE,
        validation_products,
    )

    write_jsonl(
        TEST_FILE,
        test_products,
    )

    # ---------------------------------------------------------
    # Summary
    # ---------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        "DATASET PREPARATION COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        "Training products:",
        len(
            train_products
        ),
    )

    print(
        "Validation products:",
        len(
            validation_products
        ),
    )

    print(
        "Test products:",
        len(
            test_products
        ),
    )

    print(
        "\nFILES"
    )

    print(
        "Train:",
        os.path.abspath(
            TRAIN_FILE
        )
    )

    print(
        "Validation:",
        os.path.abspath(
            VALIDATION_FILE
        )
    )

    print(
        "Test:",
        os.path.abspath(
            TEST_FILE
        )
    )

    # ---------------------------------------------------------
    # Print one example from each split.
    # ---------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TRAIN EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            train_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "VALIDATION EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            validation_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TEST EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            test_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )


if __name__ == "__main__":
    main()

Connecting to Amazon Reviews dataset...
Source: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_Appliances.jsonl
Connected successfully.
Streaming and curating products...

Finished reading dataset.
Raw rows inspected: 94327
Valid priced products: 45981

DATASET PREPARATION COMPLETE
Training products: 500
Validation products: 50
Test products: 250

FILES
Train: /content/amazon_price_train.jsonl
Validation: /content/amazon_price_validation.jsonl
Test: /content/amazon_price_test.jsonl

TRAIN EXAMPLE
{
  "description": "GE WE11X99 Htr&Hsg Asm. This is an O.E.M. Authorized part Fits with various GE brand models Oem part # WE11X99 This is an authorized aftermarket product. Fits with various GE brand models. Oem part # WE11X99 item is manufactured in United States.",
  "price": 182.56,
  "product_id": "B004XL0OIU"
}

VALIDATION EXAMPLE
{
  "description": "SkykS Door Knob Covers 2 Pcs Door Knob Protector Door Knob Covers Soft Round Door Kn

In [ ]:
!pip install -U transformers peft accelerate bitsandbytes>=0.46.1 trl

In [ ]:
import json

import torch

from datasets import load_dataset

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from trl import (
    SFTConfig,
    SFTTrainer,
)

# !pip install -U transformers peft accelerate bitsandbytes>=0.46.1

# =============================================================
# CONFIGURATION
# =============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

TRAIN_FILE = "amazon_price_train.jsonl"

VALIDATION_FILE = (
    "amazon_price_validation.jsonl"
)

OUTPUT_DIR = (
    "./qwen-price-training"
)

FINAL_ADAPTER_DIR = (
    "./qwen-price-lora-final"
)

METRICS_FILE = (
    "qwen-price-training-metrics.json"
)


# -------------------------------------------------------------
# Training settings
# -------------------------------------------------------------

NUM_EPOCHS = 3

# Use only one physical example at a time.
# This keeps GPU memory usage conservative on T4.
TRAIN_BATCH_SIZE = 1

EVAL_BATCH_SIZE = 1

# 1 physical example × 4 accumulation steps
# = effective batch size 4
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 1e-4

MAX_LENGTH = 256


# -------------------------------------------------------------
# LoRA settings
# -------------------------------------------------------------

LORA_RANK = 16

LORA_ALPHA = 32

LORA_DROPOUT = 0.05


# =============================================================
# BUILD CHAT TRAINING EXAMPLE
# =============================================================

def build_messages(example):

    price = example["price"]

    return {
        "messages": [
            {
                "role": "system",
                "content":
                    "You estimate product prices. "
                    "Reply only with the price "
                    "in US dollars.",
            },
            {
                "role": "user",
                "content":
                    example["description"],
            },
            {
                "role": "assistant",
                "content":
                    f"${price:.2f}",
            },
        ]
    }


# =============================================================
# LOAD DATASETS
# =============================================================

def load_training_datasets():

    train_dataset = load_dataset(
        "json",
        data_files=TRAIN_FILE,
        split="train",
    )

    validation_dataset = load_dataset(
        "json",
        data_files=VALIDATION_FILE,
        split="train",
    )

    print(
        "Raw training examples:",
        len(train_dataset)
    )

    print(
        "Raw validation examples:",
        len(validation_dataset)
    )

    # ---------------------------------------------------------
    # Convert:
    #
    # description + price
    #
    # into:
    #
    # system/user/assistant messages
    # ---------------------------------------------------------

    train_dataset = train_dataset.map(
        build_messages,
        remove_columns=(
            train_dataset.column_names
        ),
    )

    validation_dataset = (
        validation_dataset.map(
            build_messages,
            remove_columns=(
                validation_dataset.column_names
            ),
        )
    )

    return (
        train_dataset,
        validation_dataset,
    )


# =============================================================
# TOKENIZER
# =============================================================

def load_tokenizer():

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_NAME
        )
    )

    # ---------------------------------------------------------
    # A pad token is required when batching examples.
    # ---------------------------------------------------------

    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    tokenizer.padding_side = "right"

    return tokenizer


# =============================================================
# LOAD 4-BIT MODEL
# =============================================================

def load_quantized_model():

    # ---------------------------------------------------------
    # IMPORTANT CHANGE
    #
    # Previous version:
    #
    # bnb_4bit_compute_dtype=torch.float16
    #
    # Trainer:
    #
    # fp16=True
    #
    # That activates AMP + GradScaler.
    #
    # Our T4/PyTorch environment then encounters an internal
    # BF16 gradient and GradScaler crashes.
    #
    # ---------------------------------------------------------
    #
    # New configuration:
    #
    # Weight storage:
    #
    #     4-bit NF4
    #
    # Compute:
    #
    #     FP32
    #
    # Mixed precision:
    #
    #     disabled
    #
    # ---------------------------------------------------------

    quantization_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_compute_dtype=(
                torch.float32
            ),

            bnb_4bit_use_double_quant=True,
        )
    )

    model = (
        AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,

            quantization_config=(
                quantization_config
            ),

            # One GPU:
            # put the model explicitly on GPU 0.
            device_map={
                "": 0
            },

            # Non-quantized floating modules also use FP32.
            dtype=torch.float32,
        )
    )

    # ---------------------------------------------------------
    # KV cache is useful during generation but not training.
    # ---------------------------------------------------------

    model.config.use_cache = False

    # ---------------------------------------------------------
    # Prepare quantized model for PEFT training.
    # ---------------------------------------------------------

    model = (
        prepare_model_for_kbit_training(
            model
        )
    )

    return model


# =============================================================
# ATTACH LoRA
# =============================================================

def add_lora(model):

    lora_config = LoraConfig(

        r=LORA_RANK,

        lora_alpha=LORA_ALPHA,

        lora_dropout=LORA_DROPOUT,

        bias="none",

        task_type="CAUSAL_LM",

        # QLoRA-style targeting:
        # add LoRA adapters to applicable linear layers.
        target_modules="all-linear",
    )

    model = get_peft_model(
        model,
        lora_config,
    )

    return model


# =============================================================
# FORCE TRAINABLE ADAPTERS TO FP32
# =============================================================

def cast_trainable_parameters_to_fp32(
    model
):

    parameter_count = 0

    for parameter in model.parameters():

        if parameter.requires_grad:

            parameter.data = (
                parameter.data.float()
            )

            parameter_count += (
                parameter.numel()
            )

    print(
        "Trainable parameters forced to FP32:",
        f"{parameter_count:,}",
    )


# =============================================================
# DTYPE AUDIT
# =============================================================

def print_trainable_dtypes(model):

    dtype_counts = {}

    for parameter in model.parameters():

        if not parameter.requires_grad:
            continue

        dtype_name = str(
            parameter.dtype
        )

        dtype_counts[
            dtype_name
        ] = (
            dtype_counts.get(
                dtype_name,
                0,
            )
            + parameter.numel()
        )

    print()

    print(
        "TRAINABLE PARAMETER DTYPES"
    )

    print(
        "=" * 70
    )

    for (
        dtype_name,
        count,
    ) in dtype_counts.items():

        print(
            dtype_name,
            ":",
            f"{count:,}",
        )


# =============================================================
# CREATE TRAINING CONFIGURATION
# =============================================================

def create_training_config():

    return SFTConfig(

        # -----------------------------------------------------
        # Output/checkpoints
        # -----------------------------------------------------

        output_dir=OUTPUT_DIR,

        # -----------------------------------------------------
        # Epochs
        # -----------------------------------------------------

        num_train_epochs=(
            NUM_EPOCHS
        ),

        # -----------------------------------------------------
        # Physical batch sizes
        # -----------------------------------------------------

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        # -----------------------------------------------------
        # Effective batch:
        #
        # 1 × 4 = 4
        # -----------------------------------------------------

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        # -----------------------------------------------------
        # Learning rate
        # -----------------------------------------------------

        learning_rate=(
            LEARNING_RATE
        ),

        # -----------------------------------------------------
        # CRITICAL FIX
        #
        # Disable BOTH AMP modes.
        #
        # Therefore:
        #
        # no autocast
        # no FP16 GradScaler
        # no BF16 mixed precision
        # -----------------------------------------------------

        fp16=False,

        bf16=False,

        # -----------------------------------------------------
        # Sequence length
        # -----------------------------------------------------

        max_length=(
            MAX_LENGTH
        ),

        # -----------------------------------------------------
        # Save memory by recomputing some activations.
        # -----------------------------------------------------

        gradient_checkpointing=True,

        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },

        use_cache=False,

        # -----------------------------------------------------
        # Train only against assistant output.
        # -----------------------------------------------------

        assistant_only_loss=True,

        # -----------------------------------------------------
        # Keep data flow simple for this exercise.
        # -----------------------------------------------------

        packing=False,

        # -----------------------------------------------------
        # Use standard AdamW.
        #
        # Avoid introducing another fused or 8-bit optimizer
        # while debugging dtype compatibility.
        # -----------------------------------------------------

        optim="adamw_torch",

        # -----------------------------------------------------
        # Logging
        # -----------------------------------------------------

        logging_steps=10,

        logging_first_step=True,

        # -----------------------------------------------------
        # Validation every epoch
        # -----------------------------------------------------

        eval_strategy="epoch",

        # -----------------------------------------------------
        # Checkpoint every epoch
        # -----------------------------------------------------

        save_strategy="epoch",

        save_total_limit=2,

        # -----------------------------------------------------
        # No external logger
        # -----------------------------------------------------

        report_to="none",

        seed=42,
    )


# =============================================================
# SAVE METRICS
# =============================================================

def save_metrics(
    initial_eval,
    training_metrics,
    final_eval,
):

    metrics = {
        "initial_validation":
            initial_eval,

        "training":
            training_metrics,

        "final_validation":
            final_eval,
    }

    with open(
        METRICS_FILE,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            metrics,
            file,
            indent=2,
        )


# =============================================================
# MAIN
# =============================================================

def main():

    # ---------------------------------------------------------
    # 1. GPU validation
    # ---------------------------------------------------------

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required."
        )

    gpu_name = (
        torch.cuda.get_device_name(0)
    )

    capability = (
        torch.cuda.get_device_capability(0)
    )

    print(
        "GPU:",
        gpu_name
    )

    print(
        "CUDA capability:",
        capability
    )

    # ---------------------------------------------------------
    # 2. Experiment settings
    # ---------------------------------------------------------

    effective_batch_size = (
        TRAIN_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    )

    print(
        "Effective batch size:",
        effective_batch_size
    )

    print(
        "Epochs:",
        NUM_EPOCHS
    )

    print(
        "Learning rate:",
        LEARNING_RATE
    )

    print(
        "Mixed precision:",
        "DISABLED"
    )

    print(
        "4-bit compute dtype:",
        torch.float32
    )

    # ---------------------------------------------------------
    # 3. Load dataset
    # ---------------------------------------------------------

    print(
        "\nLoading datasets..."
    )

    (
        train_dataset,
        validation_dataset,
    ) = load_training_datasets()

    print()

    print(
        "FIRST TRAINING EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        train_dataset[0]
    )

    # ---------------------------------------------------------
    # 4. Tokenizer
    # ---------------------------------------------------------

    print(
        "\nLoading tokenizer..."
    )

    tokenizer = (
        load_tokenizer()
    )

    # ---------------------------------------------------------
    # 5. Quantized base model
    # ---------------------------------------------------------

    print(
        "\nLoading Qwen in 4-bit..."
    )

    model = (
        load_quantized_model()
    )

    print(
        "Base model memory footprint:",
        round(
            model.get_memory_footprint()
            / 1024**3,
            2,
        ),
        "GB",
    )

    # ---------------------------------------------------------
    # 6. LoRA
    # ---------------------------------------------------------

    print(
        "\nAttaching LoRA..."
    )

    model = add_lora(
        model
    )

    print()

    print(
        "TRAINABLE PARAMETERS"
    )

    print(
        "=" * 70
    )

    model.print_trainable_parameters()

    # ---------------------------------------------------------
    # 7. Ensure adapters are FP32
    # ---------------------------------------------------------

    cast_trainable_parameters_to_fp32(
        model
    )

    print_trainable_dtypes(
        model
    )

    # ---------------------------------------------------------
    # 8. Training configuration
    # ---------------------------------------------------------

    training_config = (
        create_training_config()
    )

    # ---------------------------------------------------------
    # 9. Trainer
    # ---------------------------------------------------------

    trainer = SFTTrainer(

        model=model,

        args=training_config,

        train_dataset=train_dataset,

        eval_dataset=validation_dataset,

        processing_class=tokenizer,
    )

    # ---------------------------------------------------------
    # 10. Initial validation
    # ---------------------------------------------------------

    print()

    print(
        "INITIAL VALIDATION"
    )

    print(
        "=" * 70
    )

    initial_eval = (
        trainer.evaluate()
    )

    print(
        "Initial eval loss:",
        initial_eval.get(
            "eval_loss"
        )
    )

    # ---------------------------------------------------------
    # 11. Training
    # ---------------------------------------------------------

    print()

    print(
        "STARTING QLORA TRAINING"
    )

    print(
        "=" * 70
    )

    train_result = (
        trainer.train()
    )

    # ---------------------------------------------------------
    # 12. Final validation
    # ---------------------------------------------------------

    print()

    print(
        "FINAL VALIDATION"
    )

    print(
        "=" * 70
    )

    final_eval = (
        trainer.evaluate()
    )

    print(
        "Final eval loss:",
        final_eval.get(
            "eval_loss"
        )
    )

    # ---------------------------------------------------------
    # 13. Save adapter
    # ---------------------------------------------------------

    print(
        "\nSaving LoRA adapter..."
    )

    model.save_pretrained(
        FINAL_ADAPTER_DIR
    )

    tokenizer.save_pretrained(
        FINAL_ADAPTER_DIR
    )

    # ---------------------------------------------------------
    # 14. Save metrics
    # ---------------------------------------------------------

    save_metrics(
        initial_eval,
        train_result.metrics,
        final_eval,
    )

    # ---------------------------------------------------------
    # 15. Summary
    # ---------------------------------------------------------

    print()

    print(
        "=" * 70
    )

    print(
        "TRAINING COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        "Initial eval loss:",
        initial_eval.get(
            "eval_loss"
        )
    )

    print(
        "Final eval loss:",
        final_eval.get(
            "eval_loss"
        )
    )

    print(
        "Training loss:",
        train_result.metrics.get(
            "train_loss"
        )
    )

    print(
        "Adapter saved to:",
        FINAL_ADAPTER_DIR
    )

    print(
        "Metrics saved to:",
        METRICS_FILE
    )


if __name__ == "__main__":
    main()

GPU: Tesla T4
CUDA capability: (7, 5)
Effective batch size: 4
Epochs: 3
Learning rate: 0.0001
Mixed precision: DISABLED
4-bit compute dtype: torch.float32

Loading datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Raw training examples: 500
Raw validation examples: 50


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Initial eval loss: 3.629554271697998

STARTING QLORA TRAINING


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.940116,0.933203,0.986844,36060.000000,0.698500
2,0.828442,0.959232,0.826861,72120.000000,0.687438
3,0.842149,1.010203,0.763158,108180.000000,0.691344



FINAL VALIDATION


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.842149,1.010203,3,0.763158,108180.000000,0.691344


Final eval loss: 1.010203242301941

Saving LoRA adapter...

TRAINING COMPLETE
Initial eval loss: 3.629554271697998
Final eval loss: 1.010203242301941
Training loss: 0.8887409656897358
Adapter saved to: ./qwen-price-lora-final
Metrics saved to: qwen-price-training-metrics.json


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

os.makedirs(
    "/content/drive/MyDrive/llm-engineering/week-08",
    exist_ok=True,
)

In [ ]:
import shutil


# Save trained adapter
shutil.copytree(
    "/content/qwen-price-lora-final",
    "/content/drive/MyDrive/llm-engineering/week-08/qwen-price-lora-final",
    dirs_exist_ok=True,
)


# Save datasets
for filename in [
    "amazon_price_train.jsonl",
    "amazon_price_validation.jsonl",
    "amazon_price_test.jsonl",
]:

    shutil.copy(
        f"/content/{filename}",
        "/content/drive/MyDrive/llm-engineering/week-08/",
    )

In [ ]:
import os
import re

import torch

from peft import PeftModel

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# =============================================================
# CONFIGURATION
# =============================================================

MODEL_NAME = (
    "Qwen/Qwen2.5-1.5B-Instruct"
)

# If you just finished Week 7 in the same Colab runtime:
ADAPTER_PATH = (
    "/content/drive/MyDrive/"
    "llm-engineering/week-08/"
    "qwen-price-lora-final"
)

# If you later store the adapter in Google Drive,
# change the path, for example:
#
# ADAPTER_PATH = (
#     "/content/drive/MyDrive/"
#     "qwen-price-lora-final"
# )


MAX_NEW_TOKENS = 16


# =============================================================
# SPECIALIST AGENT
# =============================================================

class SpecialistAgent:

    def __init__(self):

        print(
            "Initializing SpecialistAgent..."
        )

        self.tokenizer = None
        self.model = None

        self.load_model()

    # ---------------------------------------------------------
    # Load Qwen + trained LoRA once
    # ---------------------------------------------------------

    def load_model(self):

        if not os.path.isdir(
            ADAPTER_PATH
        ):

            raise FileNotFoundError(
                f"LoRA adapter not found: "
                f"{ADAPTER_PATH}"
            )

        adapter_config = os.path.join(
            ADAPTER_PATH,
            "adapter_config.json",
        )

        if not os.path.isfile(
            adapter_config
        ):

            raise FileNotFoundError(
                "adapter_config.json not found "
                f"inside {ADAPTER_PATH}"
            )

        print(
            "Loading tokenizer..."
        )

        self.tokenizer = (
            AutoTokenizer.from_pretrained(
                MODEL_NAME
            )
        )

        if (
            self.tokenizer.pad_token
            is None
        ):

            self.tokenizer.pad_token = (
                self.tokenizer.eos_token
            )

        # -----------------------------------------------------
        # Inference is different from training.
        #
        # Our T4 training had AMP/BF16 gradient problems.
        # During inference there is:
        #
        # no backward pass
        # no gradients
        # no GradScaler
        #
        # so FP16 compute is perfectly reasonable here.
        # -----------------------------------------------------

        quantization_config = (
            BitsAndBytesConfig(

                load_in_4bit=True,

                bnb_4bit_quant_type="nf4",

                bnb_4bit_compute_dtype=(
                    torch.float16
                ),

                bnb_4bit_use_double_quant=True,
            )
        )

        print(
            "Loading Base Qwen..."
        )

        base_model = (
            AutoModelForCausalLM
            .from_pretrained(

                MODEL_NAME,

                quantization_config=(
                    quantization_config
                ),

                device_map="auto",

                dtype=torch.float16,
            )
        )

        print(
            "Loading LoRA adapter..."
        )

        self.model = (
            PeftModel.from_pretrained(

                base_model,

                ADAPTER_PATH,

                is_trainable=False,
            )
        )

        self.model.eval()

        print(
            "SpecialistAgent ready."
        )

    # ---------------------------------------------------------
    # Convert model text into numeric price
    # ---------------------------------------------------------

    def extract_price(
        self,
        response,
    ):

        match = re.search(
            r"\$?\s*(\d[\d,]*(?:\.\d+)?)",
            response,
        )

        if match is None:

            return None

        number = (
            match.group(1)
            .replace(
                ",",
                "",
            )
        )

        try:

            return float(
                number
            )

        except ValueError:

            return None

    # ---------------------------------------------------------
    # Price one product
    # ---------------------------------------------------------

    def price(
        self,
        description,
    ):

        messages = [
            {
                "role": "system",
                "content":
                    "You estimate product prices. "
                    "Reply only with the price "
                    "in US dollars.",
            },
            {
                "role": "user",
                "content":
                    description,
            },
        ]

        inputs = (
            self.tokenizer
            .apply_chat_template(

                messages,

                tokenize=True,

                add_generation_prompt=True,

                return_tensors="pt",

                return_dict=True,
            )
        )

        input_device = (
            self.model
            .get_input_embeddings()
            .weight
            .device
        )

        inputs = inputs.to(
            input_device
        )

        input_length = (
            inputs[
                "input_ids"
            ].shape[1]
        )

        with torch.no_grad():

            output = (
                self.model.generate(

                    **inputs,

                    max_new_tokens=(
                        MAX_NEW_TOKENS
                    ),

                    do_sample=False,

                    pad_token_id=(
                        self.tokenizer
                        .eos_token_id
                    ),
                )
            )

        generated_tokens = (
            output[0][
                input_length:
            ]
        )

        raw_response = (
            self.tokenizer.decode(

                generated_tokens,

                skip_special_tokens=True,
            )
            .strip()
        )

        estimated_price = (
            self.extract_price(
                raw_response
            )
        )

        return {
            "price":
                estimated_price,

            "raw_response":
                raw_response,
        }


# =============================================================
# MAIN
# =============================================================

def main():

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required."
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    # ---------------------------------------------------------
    # Model loads once here.
    # ---------------------------------------------------------

    specialist = (
        SpecialistAgent()
    )

    # ---------------------------------------------------------
    # Multiple requests reuse the same loaded model.
    # ---------------------------------------------------------

    products = [

        (
            "55 inch OLED 4K HDR Smart TV "
            "with Dolby Vision, Dolby Atmos "
            "and four HDMI ports"
        ),

        (
            "Premium stainless steel "
            "espresso machine with "
            "integrated milk frother "
            "and programmable settings"
        ),

        (
            "Wireless noise cancelling "
            "over-ear headphones with "
            "Bluetooth and 30 hour "
            "battery life"
        ),
    ]

    print()

    print(
        "=" * 70
    )

    print(
        "SPECIALIST AGENT RESULTS"
    )

    print(
        "=" * 70
    )

    for product in products:

        result = (
            specialist.price(
                product
            )
        )

        print()

        print(
            "Product:",
            product
        )

        print(
            "Estimated price:",
            result[
                "price"
            ]
        )

        print(
            "Raw response:",
            result[
                "raw_response"
            ]
        )


if __name__ == "__main__":
    main()

GPU: Tesla T4
Initializing SpecialistAgent...
Loading tokenizer...
Loading Base Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading LoRA adapter...
SpecialistAgent ready.

SPECIALIST AGENT RESULTS

Product: 55 inch OLED 4K HDR Smart TV with Dolby Vision, Dolby Atmos and four HDMI ports
Estimated price: 999.99
Raw response: $999.99

Product: Premium stainless steel espresso machine with integrated milk frother and programmable settings
Estimated price: 170.0
Raw response: $170.00

Product: Wireless noise cancelling over-ear headphones with Bluetooth and 30 hour battery life
Estimated price: 249.0
Raw response: $249.00


In [5]:
!pip install -U openai pydantic feedparser beautifulsoup4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.5
    Uninstalling beautifulsoup4-4.13.5:
      Successfully uninstalled beautifulsoup4-4.13.5
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [7]:
!pip install -U openai pydantic feedparser beautifulsoup4 requests faiss-cpu numpy transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 66.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


In [5]:
import json
import os

from openai import OpenAI


# =============================================================
# CONFIGURATION
# =============================================================

MODEL = "gpt-5.1"

MAX_AGENT_STEPS = 10


# =============================================================
# SAMPLE DATA
#
# For Part 43 we deliberately isolate the Planner mechanics.
#
# Part 44 will replace these with our real:
#
# ScannerAgent
# EnsembleAgent
# MessengerAgent
# =============================================================

SAMPLE_DEALS = [
    {
        "description":
            "55 inch OLED 4K HDR Smart TV "
            "with Dolby Vision and Dolby Atmos",

        "deal_price":
            599.00,

        "url":
            "https://example.com/tv",
    },

    {
        "description":
            "Premium espresso machine with "
            "integrated milk frother and "
            "temperature control",

        "deal_price":
            179.00,

        "url":
            "https://example.com/espresso",
    },

    {
        "description":
            "Wireless noise cancelling headphones "
            "with 30 hour battery life",

        "deal_price":
            149.00,

        "url":
            "https://example.com/headphones",
    },
]


# =============================================================
# TOOL IMPLEMENTATIONS
# =============================================================

def scan_for_deals():

    print()
    print("TOOL: scan_for_deals")

    return {
        "deals":
            SAMPLE_DEALS
    }


def estimate_true_value(
    description,
):

    print()
    print(
        "TOOL: estimate_true_value"
    )

    print(
        "Product:",
        description
    )

    # ---------------------------------------------------------
    # Small deterministic lookup only for this planner lesson.
    #
    # Part 44 replaces this with EnsembleAgent.
    # ---------------------------------------------------------

    description_lower = (
        description.lower()
    )

    if "oled" in description_lower:

        estimated_value = 950.00

    elif "espresso" in description_lower:

        estimated_value = 420.00

    elif "headphones" in description_lower:

        estimated_value = 280.00

    else:

        estimated_value = 100.00

    return {
        "description":
            description,

        "estimated_true_value":
            estimated_value,
    }


def notify_user_of_deal(
    description,
    deal_price,
    estimated_true_value,
    url,
):

    print()
    print(
        "TOOL: notify_user_of_deal"
    )

    discount = (
        estimated_true_value
        - deal_price
    )

    print(
        "Description:",
        description
    )

    print(
        "Deal price:",
        deal_price
    )

    print(
        "Estimated value:",
        estimated_true_value
    )

    print(
        "Saving:",
        discount
    )

    print(
        "URL:",
        url
    )

    # ---------------------------------------------------------
    # Preview only.
    #
    # No real push notification yet.
    # ---------------------------------------------------------

    return {
        "status":
            "notification_preview_created",

        "description":
            description,

        "deal_price":
            deal_price,

        "estimated_true_value":
            estimated_true_value,

        "saving":
            discount,

        "url":
            url,
    }


# =============================================================
# TOOL DEFINITIONS
#
# These are contracts shown to GPT-5.1.
#
# Similar idea to Java interfaces / API schemas.
# =============================================================

TOOLS = [

    {
        "type":
            "function",

        "name":
            "scan_for_deals",

        "description":
            (
                "Scan for current candidate product deals. "
                "Returns product descriptions, selling prices "
                "and URLs."
            ),

        "parameters": {
            "type":
                "object",

            "properties": {},

            "required": [],

            "additionalProperties":
                False,
        },

        "strict":
            True,
    },


    {
        "type":
            "function",

        "name":
            "estimate_true_value",

        "description":
            (
                "Estimate the true fair value of one product. "
                "Use this for candidate deals before deciding "
                "which deal is best."
            ),

        "parameters": {
            "type":
                "object",

            "properties": {

                "description": {
                    "type":
                        "string",

                    "description":
                        (
                            "Full product description "
                            "from the scanner."
                        ),
                }
            },

            "required": [
                "description"
            ],

            "additionalProperties":
                False,
        },

        "strict":
            True,
    },


    {
        "type":
            "function",

        "name":
            "notify_user_of_deal",

        "description":
            (
                "Notify the user about the single best bargain. "
                "Call this at most once, after evaluating the "
                "candidate products."
            ),

        "parameters": {
            "type":
                "object",

            "properties": {

                "description": {
                    "type":
                        "string"
                },

                "deal_price": {
                    "type":
                        "number"
                },

                "estimated_true_value": {
                    "type":
                        "number"
                },

                "url": {
                    "type":
                        "string"
                },
            },

            "required": [
                "description",
                "deal_price",
                "estimated_true_value",
                "url",
            ],

            "additionalProperties":
                False,
        },

        "strict":
            True,
    },
]


# =============================================================
# TOOL ROUTER
#
# This maps LLM tool names to actual Python functions.
# =============================================================

def execute_tool(
    name,
    arguments,
):

    if name == "scan_for_deals":

        return scan_for_deals()


    if name == "estimate_true_value":

        return estimate_true_value(
            **arguments
        )


    if name == "notify_user_of_deal":

        return notify_user_of_deal(
            **arguments
        )


    raise ValueError(
        f"Unknown tool: {name}"
    )


# =============================================================
# AUTONOMOUS PLANNER AGENT
# =============================================================

class AutonomousPlannerAgent:

    def __init__(self):

        self.client = OpenAI()


    def run(self):

        # -----------------------------------------------------
        # This list becomes the agent's working conversation.
        # -----------------------------------------------------

        conversation = [

            {
                "role":
                    "system",

                "content":
                    (
                        "You are a deal-finding planner agent. "
                        "Your goal is to find the single best "
                        "bargain and notify the user once. "
                        "\n\n"
                        "First scan for deals. "
                        "Estimate the true value of candidate "
                        "products. Compare estimated value with "
                        "selling price. "
                        "Choose the deal with the largest "
                        "meaningful saving. "
                        "Then notify the user about that one "
                        "deal only. "
                        "After notification, finish."
                    ),
            },

            {
                "role":
                    "user",

                "content":
                    (
                        "Find the best bargain currently "
                        "available and notify me."
                    ),
            },
        ]


        # -----------------------------------------------------
        # AGENT LOOP
        # -----------------------------------------------------

        for step in range(
            1,
            MAX_AGENT_STEPS + 1,
        ):

            print()
            print(
                "=" * 70
            )

            print(
                "AGENT STEP",
                step
            )

            print(
                "=" * 70
            )

            response = (
                self.client.responses.create(

                    model=MODEL,

                    input=conversation,

                    tools=TOOLS,
                )
            )


            # -------------------------------------------------
            # Important:
            #
            # preserve everything GPT produced.
            #
            # Reasoning models may return items that need to
            # remain in the continuing interaction.
            # -------------------------------------------------

            conversation += (
                response.output
            )


            # -------------------------------------------------
            # Find requested function calls.
            # -------------------------------------------------

            function_calls = [

                item

                for item in response.output

                if item.type
                == "function_call"
            ]


            # -------------------------------------------------
            # No tool call means the agent has finished.
            # -------------------------------------------------

            if not function_calls:

                print()
                print(
                    "AGENT FINISHED"
                )

                print(
                    response.output_text
                )

                return


            # -------------------------------------------------
            # Execute every requested tool.
            #
            # The API may return more than one function call.
            # -------------------------------------------------

            for tool_call in function_calls:

                tool_name = (
                    tool_call.name
                )

                arguments = (
                    json.loads(
                        tool_call.arguments
                    )
                )

                print()

                print(
                    "Planner requested:",
                    tool_name
                )

                print(
                    "Arguments:",
                    arguments
                )


                # ---------------------------------------------
                # OUR APPLICATION executes the function.
                # GPT does not execute Python directly.
                # ---------------------------------------------

                result = (
                    execute_tool(
                        tool_name,
                        arguments,
                    )
                )


                print(
                    "Tool result:",
                    result
                )


                # ---------------------------------------------
                # Feed observation back to the model.
                #
                # call_id correlates:
                #
                # function request
                #       ↕
                # function result
                # ---------------------------------------------

                conversation.append(
                    {
                        "type":
                            "function_call_output",

                        "call_id":
                            tool_call.call_id,

                        "output":
                            json.dumps(
                                result
                            ),
                    }
                )


        raise RuntimeError(
            "Planner exceeded maximum "
            f"{MAX_AGENT_STEPS} steps."
        )


# =============================================================
# MAIN
# =============================================================

def main():

    if not os.environ.get(
        "OPENAI_API_KEY"
    ):

        raise RuntimeError(
            "OPENAI_API_KEY is not set."
        )

    planner = (
        AutonomousPlannerAgent()
    )

    planner.run()


if __name__ == "__main__":
    main()


AGENT STEP 1

Planner requested: scan_for_deals
Arguments: {}

TOOL: scan_for_deals
Tool result: {'deals': [{'description': '55 inch OLED 4K HDR Smart TV with Dolby Vision and Dolby Atmos', 'deal_price': 599.0, 'url': 'https://example.com/tv'}, {'description': 'Premium espresso machine with integrated milk frother and temperature control', 'deal_price': 179.0, 'url': 'https://example.com/espresso'}, {'description': 'Wireless noise cancelling headphones with 30 hour battery life', 'deal_price': 149.0, 'url': 'https://example.com/headphones'}]}

AGENT STEP 2

Planner requested: estimate_true_value
Arguments: {'description': '55 inch OLED 4K HDR Smart TV with Dolby Vision and Dolby Atmos'}

TOOL: estimate_true_value
Product: 55 inch OLED 4K HDR Smart TV with Dolby Vision and Dolby Atmos
Tool result: {'description': '55 inch OLED 4K HDR Smart TV with Dolby Vision and Dolby Atmos', 'estimated_true_value': 950.0}

Planner requested: estimate_true_value
Arguments: {'description': 'Premium es

In [6]:
import json
import os
import random
import re

import requests


# -------------------------------------------------------------
# Configuration
# -------------------------------------------------------------

DATA_URL = (
    "https://huggingface.co/datasets/"
    "McAuley-Lab/Amazon-Reviews-2023/"
    "resolve/main/raw/meta_categories/"
    "meta_Appliances.jsonl"
)


TRAIN_SIZE = 500
VALIDATION_SIZE = 50
TEST_SIZE = 250


TOTAL_REQUIRED = (
    TRAIN_SIZE
    + VALIDATION_SIZE
    + TEST_SIZE
)


MIN_PRICE = 0.50
MAX_PRICE = 999.49


MAX_DESCRIPTION_CHARS = 2500


RANDOM_SEED = 42


TRAIN_FILE = "amazon_price_train.jsonl"

VALIDATION_FILE = (
    "amazon_price_validation.jsonl"
)

TEST_FILE = "amazon_price_test.jsonl"


# -------------------------------------------------------------
# Parse price
# -------------------------------------------------------------

def parse_price(value):

    if value is None:
        return None

    text = str(value).strip()

    if not text:
        return None

    text = text.replace(
        "$",
        "",
    )

    text = text.replace(
        ",",
        "",
    )

    match = re.search(
        r"\d+(?:\.\d+)?",
        text,
    )

    if not match:
        return None

    try:

        return float(
            match.group()
        )

    except ValueError:

        return None


# -------------------------------------------------------------
# Clean text
# -------------------------------------------------------------

def clean_text(value):

    if value is None:
        return ""

    text = str(value)

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


# -------------------------------------------------------------
# Convert list/string into normal text
# -------------------------------------------------------------

def join_text(value):

    if value is None:
        return ""

    if isinstance(
        value,
        list,
    ):

        cleaned_items = []

        for item in value:

            cleaned = clean_text(
                item
            )

            if cleaned:

                cleaned_items.append(
                    cleaned
                )

        return " ".join(
            cleaned_items
        )

    return clean_text(
        value
    )


# -------------------------------------------------------------
# Build product text
#
# We use:
#
# title
# +
# features
# +
# description
# -------------------------------------------------------------

def build_product_text(product):

    title = clean_text(
        product.get(
            "title"
        )
    )

    features = join_text(
        product.get(
            "features"
        )
    )

    description = join_text(
        product.get(
            "description"
        )
    )

    parts = []

    if title:

        parts.append(
            title
        )

    if features:

        parts.append(
            features
        )

    if description:

        parts.append(
            description
        )

    combined = " ".join(
        parts
    )

    combined = clean_text(
        combined
    )

    # ---------------------------------------------------------
    # Character-level truncation.
    #
    # Token-level truncation happens later during SFT.
    # ---------------------------------------------------------

    return combined[
        :MAX_DESCRIPTION_CHARS
    ]


# -------------------------------------------------------------
# Curate one Amazon product
# -------------------------------------------------------------

def curate_product(product):

    price = parse_price(
        product.get(
            "price"
        )
    )

    if price is None:

        return None

    if not (
        MIN_PRICE
        <= price
        <= MAX_PRICE
    ):

        return None

    description = build_product_text(
        product
    )

    # Very short descriptions contain little useful signal.
    if len(description) < 40:

        return None

    product_id = product.get(
        "parent_asin"
    )

    if product_id is None:

        product_id = ""

    return {
        "description": description,

        "price": round(
            price,
            2,
        ),

        "product_id": str(
            product_id
        ),
    }


# -------------------------------------------------------------
# Write JSONL
# -------------------------------------------------------------

def write_jsonl(
    filename,
    products,
):

    with open(
        filename,
        "w",
        encoding="utf-8",
    ) as file:

        for product in products:

            json_line = json.dumps(
                product,
                ensure_ascii=False,
            )

            file.write(
                json_line
                + "\n"
            )


# -------------------------------------------------------------
# Reservoir sampling
#
# WHY?
#
# We do NOT want to:
#
# 1. Load the complete 285 MB dataset into memory.
# 2. Simply take the first 800 products.
#
# Reservoir sampling lets us read the dataset as a stream while
# keeping only 800 randomly selected valid products in memory.
# -------------------------------------------------------------

def collect_products():

    rng = random.Random(
        RANDOM_SEED
    )

    sampled_products = []

    valid_count = 0
    raw_count = 0

    # Metadata should normally contain one row per parent ASIN,
    # but we still guard against duplicates.
    seen_product_ids = set()

    print(
        "Connecting to Amazon Reviews dataset..."
    )

    print(
        "Source:",
        DATA_URL
    )

    # ---------------------------------------------------------
    # stream=True:
    #
    # requests does not download the entire response into
    # memory before we start processing it.
    # ---------------------------------------------------------

    with requests.get(
        DATA_URL,
        stream=True,
        timeout=300,
    ) as response:

        response.raise_for_status()

        print(
            "Connected successfully."
        )

        print(
            "Streaming and curating products..."
        )

        # -----------------------------------------------------
        # Each physical line is one JSON product.
        # -----------------------------------------------------

        for line in response.iter_lines(
            decode_unicode=True
        ):

            if not line:
                continue

            raw_count += 1

            try:

                raw_product = json.loads(
                    line
                )

            except json.JSONDecodeError:

                continue

            curated = curate_product(
                raw_product
            )

            if curated is None:

                continue

            product_id = curated[
                "product_id"
            ]

            if product_id:

                if (
                    product_id
                    in seen_product_ids
                ):

                    continue

                seen_product_ids.add(
                    product_id
                )

            # -------------------------------------------------
            # Number of valid products encountered so far.
            # -------------------------------------------------

            valid_count += 1

            # -------------------------------------------------
            # First fill the reservoir with 800 products.
            # -------------------------------------------------

            if (
                len(sampled_products)
                < TOTAL_REQUIRED
            ):

                sampled_products.append(
                    curated
                )

            else:

                # ---------------------------------------------
                # Reservoir sampling:
                #
                # Choose a random position from all valid
                # examples seen so far.
                #
                # If it falls inside our reservoir size,
                # replace one existing sample.
                # ---------------------------------------------

                position = rng.randint(
                    1,
                    valid_count,
                )

                if (
                    position
                    <= TOTAL_REQUIRED
                ):

                    sampled_products[
                        position - 1
                    ] = curated

            if (
                raw_count % 50_000
                == 0
            ):

                print(
                    "Rows inspected:",
                    raw_count,
                    "| Valid products:",
                    valid_count,
                )

    print(
        "\nFinished reading dataset."
    )

    print(
        "Raw rows inspected:",
        raw_count,
    )

    print(
        "Valid priced products:",
        valid_count,
    )

    if (
        len(sampled_products)
        < TOTAL_REQUIRED
    ):

        raise RuntimeError(
            "Not enough valid products found. "
            f"Required {TOTAL_REQUIRED}, "
            f"found {len(sampled_products)}."
        )

    # ---------------------------------------------------------
    # Shuffle the final 800 selected products before splitting.
    # ---------------------------------------------------------

    rng.shuffle(
        sampled_products
    )

    return sampled_products


# -------------------------------------------------------------
# Main
# -------------------------------------------------------------

def main():

    products = collect_products()

    # ---------------------------------------------------------
    # Create splits
    # ---------------------------------------------------------

    train_end = TRAIN_SIZE

    validation_end = (
        TRAIN_SIZE
        + VALIDATION_SIZE
    )

    train_products = products[
        :train_end
    ]

    validation_products = products[
        train_end:
        validation_end
    ]

    test_products = products[
        validation_end:
    ]

    # ---------------------------------------------------------
    # Save splits
    # ---------------------------------------------------------

    write_jsonl(
        TRAIN_FILE,
        train_products,
    )

    write_jsonl(
        VALIDATION_FILE,
        validation_products,
    )

    write_jsonl(
        TEST_FILE,
        test_products,
    )

    # ---------------------------------------------------------
    # Summary
    # ---------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        "DATASET PREPARATION COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        "Training products:",
        len(
            train_products
        ),
    )

    print(
        "Validation products:",
        len(
            validation_products
        ),
    )

    print(
        "Test products:",
        len(
            test_products
        ),
    )

    print(
        "\nFILES"
    )

    print(
        "Train:",
        os.path.abspath(
            TRAIN_FILE
        )
    )

    print(
        "Validation:",
        os.path.abspath(
            VALIDATION_FILE
        )
    )

    print(
        "Test:",
        os.path.abspath(
            TEST_FILE
        )
    )

    # ---------------------------------------------------------
    # Print one example from each split.
    # ---------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TRAIN EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            train_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "VALIDATION EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            validation_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TEST EXAMPLE"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            test_products[0],
            indent=2,
            ensure_ascii=False,
        )
    )


if __name__ == "__main__":
    main()

Connecting to Amazon Reviews dataset...
Source: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_Appliances.jsonl
Connected successfully.
Streaming and curating products...

Finished reading dataset.
Raw rows inspected: 94327
Valid priced products: 45981

DATASET PREPARATION COMPLETE
Training products: 500
Validation products: 50
Test products: 250

FILES
Train: /content/amazon_price_train.jsonl
Validation: /content/amazon_price_validation.jsonl
Test: /content/amazon_price_test.jsonl

TRAIN EXAMPLE
{
  "description": "GE WE11X99 Htr&Hsg Asm. This is an O.E.M. Authorized part Fits with various GE brand models Oem part # WE11X99 This is an authorized aftermarket product. Fits with various GE brand models. Oem part # WE11X99 item is manufactured in United States.",
  "price": 182.56,
  "product_id": "B004XL0OIU"
}

VALIDATION EXAMPLE
{
  "description": "SkykS Door Knob Covers 2 Pcs Door Knob Protector Door Knob Covers Soft Round Door Kn

In [9]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
import json
import os
import re
from pathlib import Path

import feedparser
import faiss
import numpy as np
import requests
import torch

from bs4 import BeautifulSoup
from openai import OpenAI
from pydantic import BaseModel, Field
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# ============================================================
# CONFIG
# ============================================================

PLANNER_MODEL = "gpt-5.1"
FRONTIER_MODEL = "gpt-5.1"
SCANNER_MODEL = "gpt-5.1"
MESSENGER_MODEL = "gpt-5.1"

EMBEDDING_MODEL = "text-embedding-3-small"

QWEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = (
    "/content/drive/MyDrive/"
    "llm-engineering/week-08/"
    "qwen-price-lora-final"
)

INDEX_FILE = "product_price.faiss"
METADATA_FILE = "product_price_metadata.json"

TOP_K = 5

SPECIALIST_WEIGHT = 0.50
FRONTIER_WEIGHT = 0.50

MAX_AGENT_STEPS = 12
MAX_SELECTED_DEALS = 5

# Keep notification safe while learning.
ENABLE_PUSH = False

PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
    "https://www.dealnews.com/f1912/Smart-Home/?rss=1",
]


# ============================================================
# MEMORY LOCATION
#
# If Google Drive is mounted, memory survives Colab timeout.
# Otherwise it survives only in the current runtime.
# ============================================================

DRIVE_MEMORY = Path(
    "/content/drive/MyDrive/"
    "llm-engineering/week-08/deal_agent_memory.json"
)

LOCAL_MEMORY = Path(
    "./deal_agent_memory.json"
)

MEMORY_FILE = (
    DRIVE_MEMORY
    if DRIVE_MEMORY.parent.exists()
    else LOCAL_MEMORY
)


# ============================================================
# DATA MODELS
# ============================================================

class Deal(BaseModel):
    product_description: str
    price: float
    url: str


class DealSelection(BaseModel):
    deals: list[Deal]


# ============================================================
# PERSISTENT MEMORY
# ============================================================

class DealMemory:

    def __init__(self, path):
        self.path = Path(path)

        self.data = {
            "seen_urls": [],
            "notified_urls": [],
        }

        self.load()

    def load(self):
        if self.path.exists():
            with open(
                self.path,
                "r",
                encoding="utf-8",
            ) as file:
                self.data = json.load(file)

        self.data.setdefault(
            "seen_urls",
            [],
        )

        self.data.setdefault(
            "notified_urls",
            [],
        )

    def save(self):
        self.path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with open(
            self.path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                self.data,
                file,
                indent=2,
            )

    def seen_urls(self):
        return set(
            self.data["seen_urls"]
        )

    def notified_urls(self):
        return set(
            self.data["notified_urls"]
        )

    def mark_seen(self, url):
        if url not in self.data["seen_urls"]:
            self.data["seen_urls"].append(url)
            self.save()

    def mark_notified(self, url):
        if url not in self.data["notified_urls"]:
            self.data["notified_urls"].append(url)

        self.mark_seen(url)
        self.save()

    def already_notified(self, url):
        return (
            url
            in self.notified_urls()
        )


# ============================================================
# COMMON PRICE PARSER
# ============================================================

def extract_price(text):

    match = re.search(
        r"\$?\s*(\d[\d,]*(?:\.\d+)?)",
        text,
    )

    if not match:
        return None

    return float(
        match.group(1)
        .replace(",", "")
    )


# ============================================================
# SPECIALIST AGENT
# ============================================================

class SpecialistAgent:

    def __init__(self):

        print(
            "Loading SpecialistAgent..."
        )

        if not os.path.isdir(
            ADAPTER_PATH
        ):
            raise FileNotFoundError(
                f"Adapter missing: {ADAPTER_PATH}"
            )

        self.tokenizer = (
            AutoTokenizer.from_pretrained(
                QWEN_MODEL
            )
        )

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = (
                self.tokenizer.eos_token
            )

        quantization = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        base = (
            AutoModelForCausalLM
            .from_pretrained(
                QWEN_MODEL,
                quantization_config=quantization,
                device_map="auto",
                dtype=torch.float16,
            )
        )

        self.model = (
            PeftModel.from_pretrained(
                base,
                ADAPTER_PATH,
                is_trainable=False,
            )
        )

        self.model.eval()

        print(
            "SpecialistAgent ready."
        )

    def price(self, description):

        messages = [
            {
                "role": "system",
                "content":
                    "You estimate product prices. "
                    "Reply only with the price in US dollars.",
            },
            {
                "role": "user",
                "content": description,
            },
        ]

        inputs = (
            self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        )

        device = (
            self.model
            .get_input_embeddings()
            .weight
            .device
        )

        inputs = inputs.to(device)

        input_length = (
            inputs["input_ids"]
            .shape[1]
        )

        with torch.no_grad():

            output = self.model.generate(
                **inputs,
                max_new_tokens=16,
                do_sample=False,
                pad_token_id=(
                    self.tokenizer
                    .eos_token_id
                ),
            )

        generated = output[0][
            input_length:
        ]

        raw = (
            self.tokenizer.decode(
                generated,
                skip_special_tokens=True,
            )
            .strip()
        )

        return extract_price(raw)


# ============================================================
# RAG KNOWLEDGE BASE
# ============================================================

class ProductKnowledgeBase:

    def __init__(self):

        self.client = OpenAI()

        self.index = None
        self.products = []

        if (
            os.path.exists(INDEX_FILE)
            and os.path.exists(METADATA_FILE)
        ):
            self.load_index()

        else:
            print(
                "RAG index not found. Building it..."
            )

            self.build_index()

    def load_products(self):

        train_file = (
            "amazon_price_train.jsonl"
        )

        if not os.path.exists(
            train_file
        ):
            raise FileNotFoundError(
                f"Missing {train_file}. "
                "Run dataset preparation first."
            )

        products = []

        with open(
            train_file,
            "r",
            encoding="utf-8",
        ) as file:

            for line in file:

                line = line.strip()

                if not line:
                    continue

                products.append(
                    json.loads(line)
                )

        return products

    def create_embeddings(
        self,
        texts,
    ):

        response = (
            self.client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=texts,
            )
        )

        return [
            item.embedding
            for item in response.data
        ]

    def build_index(self):

        self.products = (
            self.load_products()
        )

        print(
            "Products:",
            len(self.products)
        )

        all_embeddings = []

        batch_size = 100

        for start in range(
            0,
            len(self.products),
            batch_size,
        ):

            batch = self.products[
                start:
                start + batch_size
            ]

            descriptions = [
                product["description"]
                for product in batch
            ]

            print(
                "Embedding:",
                start + 1,
                "to",
                min(
                    start + batch_size,
                    len(self.products),
                ),
            )

            embeddings = (
                self.create_embeddings(
                    descriptions
                )
            )

            all_embeddings.extend(
                embeddings
            )

        matrix = np.array(
            all_embeddings,
            dtype="float32",
        )

        faiss.normalize_L2(
            matrix
        )

        self.index = (
            faiss.IndexFlatIP(
                matrix.shape[1]
            )
        )

        self.index.add(
            matrix
        )

        faiss.write_index(
            self.index,
            INDEX_FILE,
        )

        with open(
            METADATA_FILE,
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                self.products,
                file,
                ensure_ascii=False,
                indent=2,
            )

        print(
            "RAG index created."
        )

    def load_index(self):

        print(
            "Loading existing RAG index..."
        )

        self.index = (
            faiss.read_index(
                INDEX_FILE
            )
        )

        with open(
            METADATA_FILE,
            "r",
            encoding="utf-8",
        ) as file:

            self.products = (
                json.load(file)
            )

        print(
            "RAG products:",
            len(self.products)
        )

    def search(
        self,
        query,
        top_k=TOP_K,
    ):

        response = (
            self.client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=[query],
            )
        )

        embedding = (
            response.data[0]
            .embedding
        )

        vector = np.array(
            [embedding],
            dtype="float32",
        )

        faiss.normalize_L2(
            vector
        )

        similarities, indexes = (
            self.index.search(
                vector,
                top_k,
            )
        )

        results = []

        for similarity, index in zip(
            similarities[0],
            indexes[0],
        ):

            product = (
                self.products[
                    int(index)
                ]
            )

            results.append(
                {
                    "description":
                        product[
                            "description"
                        ],

                    "price":
                        float(
                            product[
                                "price"
                            ]
                        ),

                    "similarity":
                        float(
                            similarity
                        ),
                }
            )

        return results


# ============================================================
# FRONTIER AGENT
# ============================================================

class FrontierAgent:

    def __init__(
        self,
        knowledge_base,
    ):
        self.client = OpenAI()
        self.knowledge_base = (
            knowledge_base
        )

    def price(
        self,
        description,
    ):

        retrieved = (
            self.knowledge_base.search(
                description
            )
        )

        context_parts = []

        for i, product in enumerate(
            retrieved,
            start=1,
        ):
            context_parts.append(
                f"""
Comparable {i}
Price: ${product['price']:.2f}
Description: {product['description']}
"""
            )

        context = "\n".join(
            context_parts
        )

        prompt = f"""
Estimate the fair market price of the target product.

TARGET PRODUCT:
{description}

COMPARABLE PRODUCTS:
{context}

Use the comparable products as evidence.
Return ONLY the estimated price in US dollars.

Example:
$149.99
"""

        response = (
            self.client.responses.create(
                model=FRONTIER_MODEL,
                input=prompt,
            )
        )

        return extract_price(
            response.output_text.strip()
        )


# ============================================================
# ENSEMBLE AGENT
# ============================================================

class EnsembleAgent:

    def __init__(
        self,
        specialist,
        frontier,
    ):
        self.specialist = specialist
        self.frontier = frontier

    def price(
        self,
        description,
    ):

        specialist_price = (
            self.specialist.price(
                description
            )
        )

        frontier_price = (
            self.frontier.price(
                description
            )
        )

        if specialist_price is None:
            return frontier_price

        if frontier_price is None:
            return specialist_price

        result = (
            SPECIALIST_WEIGHT
            * specialist_price
            +
            FRONTIER_WEIGHT
            * frontier_price
        )

        print(
            "\nVALUATION"
        )

        print(
            "Specialist:",
            specialist_price
        )

        print(
            "Frontier:",
            frontier_price
        )

        print(
            "Ensemble:",
            round(result, 2)
        )

        return round(
            result,
            2,
        )


# ============================================================
# SCANNER AGENT
# ============================================================

class ScannerAgent:

    def __init__(
        self,
        memory,
    ):
        self.client = OpenAI()
        self.memory = memory

    @staticmethod
    def clean_html(text):

        return (
            BeautifulSoup(
                text or "",
                "html.parser",
            )
            .get_text(
                " ",
                strip=True,
            )
        )

    def fetch_deals(self):

        deals = []

        for feed_url in FEEDS:

            print(
                "Reading:",
                feed_url
            )

            feed = (
                feedparser.parse(
                    feed_url
                )
            )

            for entry in feed.entries[:10]:

                url = entry.get(
                    "link",
                    "",
                )

                if not url:
                    continue

                if (
                    url
                    in self.memory.seen_urls()
                ):
                    continue

                deals.append(
                    {
                        "title":
                            self.clean_html(
                                entry.get(
                                    "title",
                                    "",
                                )
                            ),

                        "summary":
                            self.clean_html(
                                entry.get(
                                    "summary",
                                    "",
                                )
                            ),

                        "url":
                            url,
                    }
                )

        return deals

    def scan(self):

        raw_deals = (
            self.fetch_deals()
        )

        if not raw_deals:
            return []

        blocks = []

        for deal in raw_deals:
            blocks.append(
                f"""
Title:
{deal['title']}

Summary:
{deal['summary']}

URL:
{deal['url']}
"""
            )

        prompt = f"""
Select up to {MAX_SELECTED_DEALS} product deals.

Requirements:

- Actual selling price must be clear.
- "$100 off" is NOT a selling price.
- Product description must contain useful product details.
- Keep the original supplied URL.
- Rewrite descriptions around the product, not promotion.

Candidates:

{"---".join(blocks)}
"""

        response = (
            self.client.responses.parse(
                model=SCANNER_MODEL,
                input=prompt,
                text_format=DealSelection,
            )
        )

        result = (
            response.output_parsed
        )

        if result is None:
            return []

        deals = result.deals[
            :MAX_SELECTED_DEALS
        ]

        # Persist that we've processed these candidates.
        for deal in deals:
            self.memory.mark_seen(
                deal.url
            )

        return [
            deal.model_dump()
            for deal in deals
        ]


# ============================================================
# MESSENGER AGENT
# ============================================================

class MessengerAgent:

    def __init__(
        self,
        memory,
    ):

        self.client = OpenAI()
        self.memory = memory

        self.pushover_user = (
            os.environ.get(
                "PUSHOVER_USER"
            )
        )

        self.pushover_token = (
            os.environ.get(
                "PUSHOVER_TOKEN"
            )
        )

    def notify(
        self,
        description,
        deal_price,
        estimated_true_value,
        url,
    ):

        # ----------------------------------------------------
        # Application-level idempotency.
        # Do NOT rely on the LLM prompt alone.
        # ----------------------------------------------------

        if self.memory.already_notified(
            url
        ):

            return {
                "status":
                    "duplicate_ignored",

                "url":
                    url,
            }

        saving = (
            estimated_true_value
            - deal_price
        )

        prompt = f"""
Write a concise 2-sentence deal notification.

Product:
{description}

Selling price:
${deal_price:.2f}

Estimated fair value:
${estimated_true_value:.2f}

Estimated saving:
${saving:.2f}

Be factual.
Return only the message.
"""

        response = (
            self.client.responses.create(
                model=MESSENGER_MODEL,
                input=prompt,
            )
        )

        message = (
            response.output_text
            .strip()
        )

        # ----------------------------------------------------
        # Safe learning mode
        # ----------------------------------------------------

        if not ENABLE_PUSH:

            print()
            print("=" * 70)
            print("NOTIFICATION PREVIEW")
            print("=" * 70)

            print(message)
            print(url)

            delivery = {
                "status":
                    "preview_only"
            }

        else:

            if not self.pushover_user:
                raise RuntimeError(
                    "PUSHOVER_USER missing."
                )

            if not self.pushover_token:
                raise RuntimeError(
                    "PUSHOVER_TOKEN missing."
                )

            response = requests.post(
                PUSHOVER_URL,
                data={
                    "token":
                        self.pushover_token,

                    "user":
                        self.pushover_user,

                    "title":
                        "Deal Alert",

                    "message":
                        message,

                    "url":
                        url,

                    "url_title":
                        "View Deal",
                },
                timeout=15,
            )

            response.raise_for_status()

            delivery = (
                response.json()
            )

        # ----------------------------------------------------
        # Mark notification only after successful delivery /
        # preview.
        # ----------------------------------------------------

        self.memory.mark_notified(
            url
        )

        return {
            "status":
                "sent",

            "message":
                message,

            "saving":
                saving,

            "url":
                url,

            "delivery":
                delivery,
        }


# ============================================================
# AUTONOMOUS PLANNER
# ============================================================

class AutonomousPlannerAgent:

    def __init__(
        self,
        scanner,
        ensemble,
        messenger,
    ):

        self.client = OpenAI()

        self.scanner = scanner
        self.ensemble = ensemble
        self.messenger = messenger

    def tools(self):

        return [
            {
                "type": "function",
                "name":
                    "scan_for_deals",

                "description":
                    (
                        "Find new product deals that "
                        "have not already been processed."
                    ),

                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": [],
                    "additionalProperties":
                        False,
                },

                "strict": True,
            },

            {
                "type": "function",
                "name":
                    "estimate_true_value",

                "description":
                    (
                        "Estimate the fair market value "
                        "of a product."
                    ),

                "parameters": {
                    "type": "object",

                    "properties": {
                        "description": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "description"
                    ],

                    "additionalProperties":
                        False,
                },

                "strict": True,
            },

            {
                "type": "function",
                "name":
                    "notify_user_of_deal",

                "description":
                    (
                        "Notify the user about the single "
                        "most compelling bargain. "
                        "Call this at most once."
                    ),

                "parameters": {
                    "type": "object",

                    "properties": {
                        "description": {
                            "type": "string"
                        },

                        "deal_price": {
                            "type": "number"
                        },

                        "estimated_true_value": {
                            "type": "number"
                        },

                        "url": {
                            "type": "string"
                        },
                    },

                    "required": [
                        "description",
                        "deal_price",
                        "estimated_true_value",
                        "url",
                    ],

                    "additionalProperties":
                        False,
                },

                "strict": True,
            },
        ]

    def execute_tool(
        self,
        name,
        arguments,
    ):

        if name == "scan_for_deals":
            return {
                "deals":
                    self.scanner.scan()
            }

        if name == "estimate_true_value":

            value = (
                self.ensemble.price(
                    arguments[
                        "description"
                    ]
                )
            )

            return {
                "description":
                    arguments[
                        "description"
                    ],

                "estimated_true_value":
                    value,
            }

        if name == "notify_user_of_deal":

            return (
                self.messenger.notify(
                    **arguments
                )
            )

        raise ValueError(
            f"Unknown tool: {name}"
        )

    def run(self):

        conversation = [
            {
                "role": "system",
                "content":
                    (
                        "You are an autonomous deal-finding "
                        "planner. "
                        "First scan for new deals. "
                        "Estimate the fair value of the useful "
                        "candidate products. "
                        "Compare each selling price with its "
                        "estimated fair value. "
                        "Select the single most compelling "
                        "bargain. "
                        "Notify the user once about that deal. "
                        "If there are no suitable new deals, "
                        "finish without notification."
                    ),
            },

            {
                "role": "user",
                "content":
                    (
                        "Find the best new bargain "
                        "and notify me."
                    ),
            },
        ]

        notification_sent = False

        for step in range(
            1,
            MAX_AGENT_STEPS + 1,
        ):

            print()
            print(
                f"===== PLANNER STEP {step} ====="
            )

            response = (
                self.client.responses.create(
                    model=PLANNER_MODEL,
                    input=conversation,
                    tools=self.tools(),
                )
            )

            conversation += (
                response.output
            )

            function_calls = [
                item
                for item in response.output
                if item.type
                == "function_call"
            ]

            if not function_calls:

                print()
                print(
                    "Planner finished:"
                )

                print(
                    response.output_text
                )

                return

            for tool_call in function_calls:

                name = tool_call.name

                arguments = (
                    json.loads(
                        tool_call.arguments
                    )
                )

                print(
                    "\nPlanner requested:",
                    name,
                )

                # ------------------------------------------------
                # Extra deterministic guardrail.
                #
                # Even if GPT asks twice, only allow one notify
                # call in this planner execution.
                # ------------------------------------------------

                if (
                    name
                    == "notify_user_of_deal"
                    and notification_sent
                ):

                    result = {
                        "status":
                            "second_notification_blocked"
                    }

                else:

                    result = (
                        self.execute_tool(
                            name,
                            arguments,
                        )
                    )

                    if (
                        name
                        == "notify_user_of_deal"
                        and result.get(
                            "status"
                        )
                        in {
                            "sent",
                            "duplicate_ignored",
                        }
                    ):
                        notification_sent = True

                print(
                    "Tool result:",
                    result,
                )

                conversation.append(
                    {
                        "type":
                            "function_call_output",

                        "call_id":
                            tool_call.call_id,

                        "output":
                            json.dumps(
                                result
                            ),
                    }
                )

        raise RuntimeError(
            "Planner exceeded maximum steps."
        )


# ============================================================
# MAIN
# ============================================================

def main():

    if not os.environ.get(
        "OPENAI_API_KEY"
    ):
        raise RuntimeError(
            "OPENAI_API_KEY is not set."
        )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Colab GPU is required "
            "for SpecialistAgent."
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "Memory file:",
        MEMORY_FILE
    )

    memory = DealMemory(
        MEMORY_FILE
    )

    knowledge_base = (
        ProductKnowledgeBase()
    )

    specialist = (
        SpecialistAgent()
    )

    frontier = (
        FrontierAgent(
            knowledge_base
        )
    )

    ensemble = (
        EnsembleAgent(
            specialist,
            frontier,
        )
    )

    scanner = (
        ScannerAgent(
            memory
        )
    )

    messenger = (
        MessengerAgent(
            memory
        )
    )

    planner = (
        AutonomousPlannerAgent(
            scanner=scanner,
            ensemble=ensemble,
            messenger=messenger,
        )
    )

    planner.run()

    print()
    print(
        "=" * 70
    )

    print(
        "PERSISTED MEMORY"
    )

    print(
        "=" * 70
    )

    print(
        json.dumps(
            memory.data,
            indent=2,
        )
    )


if __name__ == "__main__":
    main()

GPU: Tesla T4
Memory file: /content/drive/MyDrive/llm-engineering/week-08/deal_agent_memory.json
Loading existing RAG index...
RAG products: 500
Loading SpecialistAgent...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

SpecialistAgent ready.

===== PLANNER STEP 1 =====

Planner requested: scan_for_deals
Reading: https://www.dealnews.com/c142/Electronics/?rss=1
Reading: https://www.dealnews.com/c39/Computers/?rss=1
Reading: https://www.dealnews.com/f1912/Smart-Home/?rss=1
Tool result: {'deals': [{'product_description': 'Jaszdot 32" floor-standing digital signage display with interactive touchscreen, Android OS with Google apps support, USB/HDMI/WiFi inputs, split‑screen multi-content playback, and a 600‑nit ultra-clear IPS panel designed for high-traffic retail, restaurant, or office environments (model CA-32B).', 'price': 539.4, 'url': 'https://www.dealnews.com/products/Jaszdot/Jaszdot-32-Digital-Signage-Display/537735.html?iref=rss-c142'}, {'product_description': 'YLZKIX gaming desktop powered by an AMD Ryzen 5 5500 6-core/12-thread CPU (3.6GHz, boost up to 4.2GHz), AMD Radeon RX 7600 8GB graphics card, 16GB DDR4-3200 RAM, and a 1TB M.2 NVMe SSD, with WiFi 6, Bluetooth, 550W power supply, 90mm tower